<a href="https://colab.research.google.com/github/msdurk/masters/blob/main/machinewars_vs_trad.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip -q install transformers datasets accelerate scikit-learn pandas numpy

In [2]:
import os
import re
import json
import random
import numpy as np
import pandas as pd
import torch

from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score,
    precision_recall_fscore_support,
    roc_auc_score
)

from datasets import Dataset
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    DataCollatorWithPadding,
    TrainingArguments,
    Trainer,
    set_seed,
)

In [3]:
SEED = 42

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

set_seed(SEED)

device = "cuda" if torch.cuda.is_available() else "cpu"
print("Using device:", device)

Using device: cuda


In [4]:
from google.colab import files
uploaded = files.upload()

Saving machinewars_filtered_emails.json to machinewars_filtered_emails.json
Saving CEAS_08_cleaned.csv to CEAS_08_cleaned.csv
Saving Nazario_cleaned.csv to Nazario_cleaned.csv
Saving Nigerian_Fraud_cleaned.csv to Nigerian_Fraud_cleaned.csv
Saving SpamAssasin_cleaned.csv to SpamAssasin_cleaned.csv


In [5]:
import pandas as pd
import json
from pathlib import Path


# ---------------------------------
# 1. Helpers
# ---------------------------------
def safe_str(x):
    if pd.isna(x):
        return ""
    return str(x).strip()


def build_text_subject_body(row):
    subject = safe_str(row.get("subject", ""))
    body = safe_str(row.get("body", ""))
    return f"{subject}\n\n{body}".strip()


# ---------------------------------
# 2. Label handling
# ---------------------------------
def normalize_machinewars_label(label, spam_as_phishing=False):
    """
    MachineWars:
      Phishing -> 1
      Legitimate/Valid/Ham -> 0
      Spam -> 1 if spam_as_phishing=True else excluded
    """
    if pd.isna(label):
        return None

    label = str(label).strip().lower()

    if label == "phishing":
        return 1

    if label in {"legitimate", "valid", "ham", "benign", "safe"}:
        return 0

    if label == "spam":
        return 1 if spam_as_phishing else None

    return None


def normalize_test_label(label):
    """
    For CEAS-style test sets.
    Spam is excluded here unless you explicitly want otherwise.
    """
    if pd.isna(label):
        return None

    label = str(label).strip().lower()

    if label == "phishing":
        return 1

    if label in {"legitimate", "valid", "ham", "benign", "safe"}:
        return 0

    return None


# ---------------------------------
# 3. MachineWars loader
# ---------------------------------
def load_machinewars(json_path_or_list, spam_as_phishing=False, dataset_name="machinewars"):
    """
    Expected MachineWars fields:
      sender, subject, body, type, url
    """
    if isinstance(json_path_or_list, (str, Path)):
        with open(json_path_or_list, "r", encoding="utf-8") as f:
            data = json.load(f)
    else:
        data = json_path_or_list

    df = pd.DataFrame(data).copy()

    # rename type -> label
    if "type" in df.columns:
        df = df.rename(columns={"type": "label"})

    # ensure required columns exist
    for col in ["subject", "body", "label"]:
        if col not in df.columns:
            df[col] = ""

    # keep optional columns if they exist
    if "sender" not in df.columns:
        df["sender"] = ""

    if "url" not in df.columns:
        df["url"] = ""

    # flatten url list to string
    df["url"] = df["url"].apply(
        lambda x: " | ".join(x) if isinstance(x, list) else safe_str(x)
    )

    df["dataset"] = dataset_name
    df["label_raw"] = df["label"].astype(str).str.strip().str.lower()
    df["label_id"] = df["label_raw"].apply(
        lambda x: normalize_machinewars_label(x, spam_as_phishing=spam_as_phishing)
    )

    # drop excluded rows
    df = df[df["label_id"].notna()].copy()
    df["label_id"] = df["label_id"].astype(int)

    # normalized binary label name
    df["label"] = df["label_id"].map({0: "legitimate", 1: "phishing"})

    # only subject + body for model text
    df["text"] = df.apply(build_text_subject_body, axis=1)

    # final schema
    df = df[
        ["dataset", "sender", "subject", "body", "url", "label_raw", "label", "label_id", "text"]
    ]

    return df


# ---------------------------------
# 4. CEAS-style test loader
# ---------------------------------
def load_ceas_style_csv(csv_path, dataset_name=None):
    """
    Assumes CEAS-style columns similar to:
      subject, body, label
    """
    csv_path = Path(csv_path)
    if dataset_name is None:
        dataset_name = csv_path.stem

    df = pd.read_csv(csv_path).copy()

    rename_map = {
        "Subject": "subject",
        "Body": "body",
        "Label": "label",
        "type": "label",
    }
    df = df.rename(columns=rename_map)

    for col in ["subject", "body", "label"]:
        if col not in df.columns:
            df[col] = ""

    if "sender" not in df.columns:
        df["sender"] = ""

    if "url" not in df.columns:
        df["url"] = ""

    df["dataset"] = dataset_name
    df["label_raw"] = df["label"].astype(str).str.strip().str.lower()
    df["label_id"] = df["label_raw"].apply(normalize_test_label)

    # drop non-binary rows
    df = df[df["label_id"].notna()].copy()
    df["label_id"] = df["label_id"].astype(int)
    df["label"] = df["label_id"].map({0: "legitimate", 1: "phishing"})
    df["text"] = df.apply(build_text_subject_body, axis=1)

    df = df[
        ["dataset", "sender", "subject", "body", "url", "label_raw", "label", "label_id", "text"]
    ]

    return df


# ---------------------------------
# 5. Create the two MachineWars versions
# ---------------------------------
machinewars_path = "machinewars_filtered_emails.json"

# Version A: spam merged into phishing
machinewars_spam_as_phishing_df = load_machinewars(
    machinewars_path,
    spam_as_phishing=True,
    dataset_name="machinewars"
)

# Version B: spam removed
machinewars_no_spam_df = load_machinewars(
    machinewars_path,
    spam_as_phishing=False,
    dataset_name="machinewars"
)

print("MachineWars: spam merged into phishing")
print(machinewars_spam_as_phishing_df["label"].value_counts())
print(machinewars_spam_as_phishing_df.head(3))

print("\nMachineWars: spam removed")
print(machinewars_no_spam_df["label"].value_counts())
print(machinewars_no_spam_df.head(3))


# ---------------------------------
# 6. Load the 4 CEAS-style test datasets
# ---------------------------------
test_paths = [
    "CEAS_08_cleaned.csv",
    "Nazario_cleaned.csv",
    "Nigerian_Fraud_cleaned.csv",
    "SpamAssasin_cleaned.csv",
]

test_dfs = [load_ceas_style_csv(p) for p in test_paths]

for i, df in enumerate(test_dfs, 1):
    print(f"\nTest dataset {i}:")
    print(df["label"].value_counts())
    print(df.head(2))

MachineWars: spam merged into phishing
label
phishing      13200
legitimate     6600
Name: count, dtype: int64
       dataset                                             sender  \
0  machinewars      Dropbox Security <noreply@dropbox-secure.net>   
1  machinewars  Google Drive Security <security-alert@google-d...   
2  machinewars  Microsoft OneDrive Security <noreply@microsoft...   

                                             subject  \
0  Unusual Sign-in Activity Detected on Your Drop...   
1  Security Alert: New Sign-in to Your Google Dri...   
2  Important Security Notification Regarding Your...   

                                                body  \
0  Dear User,\n\nWe've detected an unusual sign-i...   
1  Google Drive Security Alert\n\nWe've noticed a...   
2  Hello sarah.smith@gmail.com,\n\nThis is an aut...   

                                                 url label_raw     label  \
0         https://dropbox-security.co/account/review  phishing  phishing   
1  https:/

In [6]:
train_df, val_df = train_test_split(
    machinewars_spam_as_phishing_df,
    test_size=0.2,
    random_state=SEED,
    stratify=machinewars_spam_as_phishing_df["label_id"]
)

In [7]:
from datasets import Dataset

train_ds = Dataset.from_pandas(
    train_df[["text", "label_id"]]
    .rename(columns={"label_id": "labels"})
    .reset_index(drop=True)
)

val_ds = Dataset.from_pandas(
    val_df[["text", "label_id"]]
    .rename(columns={"label_id": "labels"})
    .reset_index(drop=True)
)

In [8]:
MODEL_NAME = "distilbert-base-uncased"

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=2
)

config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/268M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_transform.bias    | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
pre_classifier.bias     | MISSING    | 
pre_classifier.weight   | MISSING    | 
classifier.weight       | MISSING    | 
classifier.bias         | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [9]:
import torch

device = "cuda" if torch.cuda.is_available() else "cpu"
model = model.to(device)
model.eval()

# Enable faster matmul on Ampere/A100
torch.set_float32_matmul_precision("high")

# Optional: allow TF32 on A100
torch.backends.cuda.matmul.allow_tf32 = True
torch.backends.cudnn.allow_tf32 = True

print(device)

cuda


In [10]:
def tokenize_function(batch):
    return tokenizer(
        batch["text"],
        truncation=True,
        max_length=512
    )

train_ds = train_ds.map(tokenize_function, batched=True)
val_ds = val_ds.map(tokenize_function, batched=True)

train_ds = train_ds.remove_columns(["text"])
val_ds = val_ds.remove_columns(["text"])

data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

Map:   0%|          | 0/15840 [00:00<?, ? examples/s]

Map:   0%|          | 0/3960 [00:00<?, ? examples/s]

In [11]:
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    probs = torch.softmax(torch.tensor(logits), dim=-1).numpy()
    preds = np.argmax(probs, axis=1)

    precision, recall, f1, _ = precision_recall_fscore_support(
        labels,
        preds,
        average="binary",
        zero_division=0
    )
    acc = accuracy_score(labels, preds)

    try:
        roc_auc = roc_auc_score(labels, probs[:, 1])
    except:
        roc_auc = float("nan")

    return {
        "accuracy": acc,
        "precision": precision,
        "recall": recall,
        "f1": f1,
        "roc_auc": roc_auc,
    }

In [12]:
training_args = TrainingArguments(
    output_dir="/content/bert_phishing_output",
    eval_strategy="epoch",
    save_strategy="epoch",
    logging_strategy="steps",
    logging_steps=50,
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=32,
    num_train_epochs=3,
    weight_decay=0.01,
    load_best_model_at_end=True,
    metric_for_best_model="f1",
    greater_is_better=True,
    save_total_limit=2,
    report_to="none",
    fp16=torch.cuda.is_available(),
)

In [13]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_ds,
    eval_dataset=val_ds,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
)

In [14]:
trainer.train()

Epoch,Training Loss,Validation Loss,Accuracy,Precision,Recall,F1,Roc Auc
1,0.115423,0.103351,0.965657,0.962676,0.986742,0.974560,0.992051
2,0.068100,0.095896,0.975253,0.977102,0.985985,0.981523,0.995724
3,0.052369,0.089490,0.977525,0.977894,0.988636,0.983236,0.996945


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['distilbert.embeddings.LayerNorm.weight', 'distilbert.embeddings.LayerNorm.bias'].
There were unexpected keys in the checkpoint model loaded: ['distilbert.embeddings.LayerNorm.beta', 'distilbert.embeddings.LayerNorm.gamma'].


TrainOutput(global_step=2970, training_loss=0.0963696556962299, metrics={'train_runtime': 155.9298, 'train_samples_per_second': 304.753, 'train_steps_per_second': 19.047, 'total_flos': 6290744294774784.0, 'train_loss': 0.0963696556962299, 'epoch': 3.0})

In [ ]:
try:
    model = torch.compile(model)
    print("Model compiled successfully.")
except Exception as e:
    print("torch.compile not available or failed:", e)

Model compiled successfully.


In [ ]:
print("=== Validation (in-domain) ===")
val_metrics = trainer.evaluate()
print(val_metrics)

=== Validation (in-domain) ===


{'eval_loss': 0.042487502098083496, 'eval_accuracy': 0.9916666666666667, 'eval_precision': 0.9831378299120235, 'eval_recall': 0.9925980754996299, 'eval_f1': 0.9878453038674033, 'eval_roc_auc': 0.9993818017061592, 'eval_runtime': 4.7355, 'eval_samples_per_second': 836.232, 'eval_steps_per_second': 26.185, 'epoch': 3.0}


In [25]:
from datasets import Dataset

all_results = {}

for i, test_df in enumerate(test_dfs, start=1):
    test_name = test_df["dataset"].iloc[0] if "dataset" in test_df.columns else f"test_{i}"

    test_ds = Dataset.from_pandas(
        test_df[["text", "label_id"]]
        .rename(columns={"label_id": "labels"})
        .reset_index(drop=True)
    )

    test_ds = test_ds.map(tokenize_function, batched=True)
    test_ds = test_ds.remove_columns(["text"])

    metrics = trainer.evaluate(eval_dataset=test_ds)
    all_results[test_name] = metrics

    print(f"\nResults for {test_name}:")
    print(metrics)

Map:   0%|          | 0/39154 [00:00<?, ? examples/s]


Results for CEAS_08_cleaned:
{'eval_loss': 1.7607892751693726, 'eval_accuracy': 0.7359656740052102, 'eval_precision': 0.9621565161497669, 'eval_recall': 0.5482556542441168, 'eval_f1': 0.6984951003266449, 'eval_roc_auc': 0.9548186323353014, 'eval_runtime': 41.2973, 'eval_samples_per_second': 948.102, 'eval_steps_per_second': 29.639, 'epoch': 3.0}


Map:   0%|          | 0/1565 [00:00<?, ? examples/s]


Results for Nazario_cleaned:
{'eval_loss': 0.1047729030251503, 'eval_accuracy': 0.9776357827476039, 'eval_precision': 1.0, 'eval_recall': 0.9776357827476039, 'eval_f1': 0.9886914378029079, 'eval_roc_auc': nan, 'eval_runtime': 1.7226, 'eval_samples_per_second': 908.534, 'eval_steps_per_second': 28.446, 'epoch': 3.0}


/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_ranking.py:379: UndefinedMetricWarning: Only one class is present in y_true. ROC AUC score is not defined in that case.
  warnings.warn(


Map:   0%|          | 0/3332 [00:00<?, ? examples/s]

/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_ranking.py:379: UndefinedMetricWarning: Only one class is present in y_true. ROC AUC score is not defined in that case.
  warnings.warn(



Results for Nigerian_Fraud_cleaned:
{'eval_loss': 0.17072753608226776, 'eval_accuracy': 0.9606842737094838, 'eval_precision': 1.0, 'eval_recall': 0.9606842737094838, 'eval_f1': 0.9799479565283943, 'eval_roc_auc': nan, 'eval_runtime': 4.4648, 'eval_samples_per_second': 746.283, 'eval_steps_per_second': 23.517, 'epoch': 3.0}


Map:   0%|          | 0/5809 [00:00<?, ? examples/s]


Results for SpamAssasin_cleaned:
{'eval_loss': 0.5819458961486816, 'eval_accuracy': 0.9013599586847995, 'eval_precision': 0.9518547750591949, 'eval_recall': 0.7019790454016298, 'eval_f1': 0.8080402010050252, 'eval_roc_auc': 0.9596883359906708, 'eval_runtime': 6.8974, 'eval_samples_per_second': 842.207, 'eval_steps_per_second': 26.387, 'epoch': 3.0}


In [ ]:
save_dir = "/content/final_phishing_bert"

trainer.save_model(save_dir)
tokenizer.save_pretrained(save_dir)

print("Saved to:", save_dir)

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Saved to: /content/final_phishing_bert


In [26]:
def predict_email(subject, body):
    text = f"Subject: {subject}\n\nBody: {body}"

    inputs = tokenizer(
        text,
        return_tensors="pt",
        truncation=True,
        max_length=512
    )

    inputs = {k: v.to(model.device) for k, v in inputs.items()}

    model.eval()
    with torch.no_grad():
        outputs = model(**inputs)
        probs = torch.softmax(outputs.logits, dim=-1).cpu().numpy()[0]

    pred = int(np.argmax(probs))

    return {
        "predicted_label": pred,   # 1 = phishing, 0 = not phishing
        "phishing_prob": float(probs[1]),
        "not_phishing_prob": float(probs[0]),
    }

example = predict_email(
    subject="Security alert: unusual login attempt",
    body="We detected suspicious activity on your account. Please verify immediately."
)

example

{'predicted_label': 1,
 'phishing_prob': 0.9993600249290466,
 'not_phishing_prob': 0.0006399274570867419}

In [27]:
def perturb_char_noise(text, p=0.03):
    swaps = {"o": "0", "i": "1", "e": "3", "a": "@", "s": "$"}
    chars = list(text)

    for i, ch in enumerate(chars):
        if ch.lower() in swaps and random.random() < p:
            chars[i] = swaps[ch.lower()]

    return "".join(chars)

def perturb_delete_keywords(text, keywords=None):
    if keywords is None:
        keywords = ["verify", "security", "urgent", "account", "password", "login"]

    out = text
    for kw in keywords:
        out = re.sub(rf"\b{re.escape(kw)}\b", "", out, flags=re.IGNORECASE)

    out = re.sub(r"\s+", " ", out).strip()
    return out

def perturb_truncate(text, max_words=50):
    return " ".join(text.split()[:max_words])

def perturb_url_mask(text):
    return re.sub(r"https?://\S+|www\.\S+", "[LINK]", text)

def perturb_subject_only(text):
    m = re.search(r"Subject:\s*(.*?)\n\s*\nBody:", text, flags=re.DOTALL | re.IGNORECASE)
    return m.group(1).strip() if m else text

def perturb_body_only(text):
    m = re.search(r"Body:\s*(.*)$", text, flags=re.DOTALL | re.IGNORECASE)
    return m.group(1).strip() if m else text

In [28]:
def batch_predict(texts):
    preds = []
    probs_out = []

    model.eval()

    for text in texts:
        inputs = tokenizer(
            text,
            return_tensors="pt",
            truncation=True,
            max_length=512
        )
        inputs = {k: v.to(model.device) for k, v in inputs.items()}

        with torch.no_grad():
            outputs = model(**inputs)
            probs = torch.softmax(outputs.logits, dim=-1).cpu().numpy()[0]

        preds.append(int(np.argmax(probs)))
        probs_out.append(float(probs[1]))

    return preds, probs_out

In [29]:
def batch_predict_fast(texts, model=model, tokenizer=tokenizer, batch_size=64, max_length=512):
    preds = []
    probs_out = []

    for i in range(0, len(texts), batch_size):
        batch = texts[i:i+batch_size]

        enc = tokenizer(
            batch,
            return_tensors="pt",
            truncation=True,
            max_length=max_length,
            padding=True,
        )
        enc = {k: v.to(model.device, non_blocking=True) for k, v in enc.items()}

        with torch.inference_mode():
            with torch.autocast(device_type="cuda", dtype=torch.float16, enabled=torch.cuda.is_available()):
                outputs = model(**enc)
                probs = torch.softmax(outputs.logits, dim=-1)

        probs = probs.detach().float().cpu().numpy()
        preds.extend(np.argmax(probs, axis=1).tolist())
        probs_out.extend(probs[:, 1].tolist())

    return preds, probs_out

In [31]:
import pandas as pd
import numpy as np
from sklearn.metrics import precision_recall_fscore_support, accuracy_score, roc_auc_score


def evaluate_under_attacks(df_eval):
    """
    Expects df_eval to have at least:
      - subject
      - body
      - text
      - label_id

    Uses your existing batch_predict_fast(text_list) function, which should return:
      - y_pred: predicted class labels
      - y_prob: probability for the positive class
    """

    def safe_str(x):
        if pd.isna(x):
            return ""
        return str(x).strip()

    def join_subject_body(subject, body):
        subject = safe_str(subject)
        body = safe_str(body)
        return f"{subject}\n\n{body}".strip()

    # Row-aware attacks
    def attack_clean(row):
        return safe_str(row["text"])

    def attack_char_noise(row):
        return perturb_char_noise(safe_str(row["text"]), p=0.03)

    def attack_delete_keywords(row):
        return perturb_delete_keywords(safe_str(row["text"]))

    def attack_truncate_50w(row):
        return perturb_truncate(safe_str(row["text"]), max_words=50)

    def attack_url_mask(row):
        return perturb_url_mask(safe_str(row["text"]))

    def attack_subject_only(row):
        return safe_str(row["subject"])

    def attack_body_only(row):
        return safe_str(row["body"])

    # Optional: preserve the two-part email format while zeroing one part
    def attack_subject_plus_empty_body(row):
        return join_subject_body(row["subject"], "")

    def attack_empty_subject_plus_body(row):
        return join_subject_body("", row["body"])

    attacks = {
        "clean": attack_clean,
        "char_noise": attack_char_noise,
        "delete_keywords": attack_delete_keywords,
        "truncate_50w": attack_truncate_50w,
        "url_mask": attack_url_mask,
        "subject_only": attack_subject_only,
        "body_only": attack_body_only,
        # optional variants:
        # "subject_plus_empty_body": attack_subject_plus_empty_body,
        # "empty_subject_plus_body": attack_empty_subject_plus_body,
    }

    results = []

    y_true = df_eval["label_id"].astype(int).tolist()

    for attack_name, attack_fn in attacks.items():
        attacked_texts = df_eval.apply(attack_fn, axis=1).tolist()
        y_pred, y_prob = batch_predict_fast(attacked_texts)

        precision, recall, f1, _ = precision_recall_fscore_support(
            y_true,
            y_pred,
            average="binary",
            zero_division=0
        )
        acc = accuracy_score(y_true, y_pred)

        try:
            roc_auc = roc_auc_score(y_true, y_prob)
        except Exception:
            roc_auc = float("nan")

        results.append({
            "attack": attack_name,
            "n_samples": len(df_eval),
            "accuracy": acc,
            "precision": precision,
            "recall": recall,
            "f1": f1,
            "roc_auc": roc_auc,
        })

    return pd.DataFrame(results)

In [37]:
robustness_results = evaluate_under_attacks(val_df.reset_index(drop=True))
robustness_results = robustness_results.sort_values("f1", ascending=False)
print(robustness_results)

            attack  n_samples  accuracy  precision    recall        f1  \
0            clean       3960  0.979293   0.981551  0.987500  0.984517   
2  delete_keywords       3960  0.978283   0.982615  0.984848  0.983731   
1       char_noise       3960  0.977020   0.974674  0.991288  0.982911   
6        body_only       3960  0.975505   0.978547  0.984848  0.981688   
4         url_mask       3960  0.975253   0.969697  0.993939  0.981669   
3     truncate_50w       3960  0.936364   0.921313  0.989015  0.953964   
5     subject_only       3960  0.905051   0.887937  0.981439  0.932350   

    roc_auc  
0  0.997425  
2  0.997009  
1  0.996374  
6  0.996159  
4  0.996206  
3  0.978478  
5  0.944930  


In [38]:
all_robustness_results = {}

for i, test_df in enumerate(test_dfs, start=1):
    test_name = test_df["dataset"].iloc[0] if "dataset" in test_df.columns else f"test_{i}"
    result_df = evaluate_under_attacks(test_df.reset_index(drop=True))
    all_robustness_results[test_name] = result_df

    print(f"\nRobustness results for {test_name}")
    print(result_df.sort_values("f1", ascending=False))


Robustness results for CEAS_08_cleaned
            attack  n_samples  accuracy  precision    recall        f1  \
5     subject_only      39154  0.820121   0.914375  0.747551  0.822590   
3     truncate_50w      39154  0.762451   0.946204  0.608781  0.740883   
4         url_mask      39154  0.759641   0.957999  0.595229  0.734250   
2  delete_keywords      39154  0.736093   0.962692  0.548164  0.698562   
0            clean      39154  0.735991   0.962160  0.548301  0.698533   
1       char_noise      39154  0.733131   0.956633  0.546379  0.695515   
6        body_only      39154  0.721229   0.956547  0.524082  0.677157   

    roc_auc  
5  0.932223  
3  0.942807  
4  0.956962  
2  0.955087  
0  0.954819  
1  0.952448  
6  0.928119  


/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_ranking.py:379: UndefinedMetricWarning: Only one class is present in y_true. ROC AUC score is not defined in that case.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_ranking.py:379: UndefinedMetricWarning: Only one class is present in y_true. ROC AUC score is not defined in that case.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_ranking.py:379: UndefinedMetricWarning: Only one class is present in y_true. ROC AUC score is not defined in that case.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_ranking.py:379: UndefinedMetricWarning: Only one class is present in y_true. ROC AUC score is not defined in that case.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_ranking.py:379: UndefinedMetricWarning: Only one class is present in y_true. ROC AUC score is not defined in that case.
  warnings.warn(
/usr/local/lib/python3.12/dist


Robustness results for Nazario_cleaned
            attack  n_samples  accuracy  precision    recall        f1  \
4         url_mask       1565  0.983387        1.0  0.983387  0.991624   
1       char_noise       1565  0.980192        1.0  0.980192  0.989997   
3     truncate_50w       1565  0.978914        1.0  0.978914  0.989345   
6        body_only       1565  0.978275        1.0  0.978275  0.989018   
0            clean       1565  0.977636        1.0  0.977636  0.988691   
2  delete_keywords       1565  0.971885        1.0  0.971885  0.985742   
5     subject_only       1565  0.847284        1.0  0.847284  0.917330   

   roc_auc  
4      NaN  
1      NaN  
3      NaN  
6      NaN  
0      NaN  
2      NaN  
5      NaN  


/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_ranking.py:379: UndefinedMetricWarning: Only one class is present in y_true. ROC AUC score is not defined in that case.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_ranking.py:379: UndefinedMetricWarning: Only one class is present in y_true. ROC AUC score is not defined in that case.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_ranking.py:379: UndefinedMetricWarning: Only one class is present in y_true. ROC AUC score is not defined in that case.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_ranking.py:379: UndefinedMetricWarning: Only one class is present in y_true. ROC AUC score is not defined in that case.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_ranking.py:379: UndefinedMetricWarning: Only one class is present in y_true. ROC AUC score is not defined in that case.
  warnings.warn(
/usr/local/lib/python3.12/dist


Robustness results for Nigerian_Fraud_cleaned
            attack  n_samples  accuracy  precision    recall        f1  \
4         url_mask       3332  0.973589        1.0  0.973589  0.986618   
6        body_only       3332  0.962785        1.0  0.962785  0.981040   
0            clean       3332  0.960684        1.0  0.960684  0.979948   
3     truncate_50w       3332  0.958583        1.0  0.958583  0.978854   
2  delete_keywords       3332  0.955282        1.0  0.955282  0.977130   
1       char_noise       3332  0.954082        1.0  0.954082  0.976501   
5     subject_only       3332  0.857443        1.0  0.857443  0.923251   

   roc_auc  
4      NaN  
6      NaN  
0      NaN  
3      NaN  
2      NaN  
1      NaN  
5      NaN  

Robustness results for SpamAssasin_cleaned
            attack  n_samples  accuracy  precision    recall        f1  \
4         url_mask       5809  0.928387   0.957806  0.792782  0.867516   
3     truncate_50w       5809  0.910139   0.915855  0.766589  0.

In [ ]:
candidate_models = [
    "distilbert-base-uncased",
    "bert-base-uncased",
    "roberta-base",
]

summary_results = []

for model_name in candidate_models:
    print(f"\n===== Training {model_name} =====")

    tokenizer = AutoTokenizer.from_pretrained(model_name)
    model = AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=2)

    train_ds_tmp = Dataset.from_pandas(train_df.reset_index(drop=True))
    val_ds_tmp = Dataset.from_pandas(val_df.reset_index(drop=True))

    def tok(batch):
        return tokenizer(batch["text"], truncation=True, max_length=512)

    train_ds_tmp = train_ds_tmp.map(tok, batched=True)
    val_ds_tmp = val_ds_tmp.map(tok, batched=True)

    collator = DataCollatorWithPadding(tokenizer=tokenizer)

    args = TrainingArguments(
        output_dir=f"/content/{model_name.replace('/', '_')}",
        eval_strategy="epoch",
        save_strategy="no",
        logging_strategy="steps",
        logging_steps=50,
        learning_rate=2e-5,
        per_device_train_batch_size=16,
        per_device_eval_batch_size=32,
        num_train_epochs=2,
        weight_decay=0.01,
        report_to="none",
        fp16=torch.cuda.is_available(),
    )

    trainer = Trainer(
        model=model,
        args=args,
        train_dataset=train_ds_tmp,
        eval_dataset=val_ds_tmp,
        data_collator=collator,
        compute_metrics=compute_metrics,
    )

    trainer.train()
    metrics = trainer.evaluate()

    summary_results.append({
        "model": model_name,
        "accuracy": metrics.get("eval_accuracy"),
        "precision": metrics.get("eval_precision"),
        "recall": metrics.get("eval_recall"),
        "f1": metrics.get("eval_f1"),
        "roc_auc": metrics.get("eval_roc_auc"),
    })

results_df = pd.DataFrame(summary_results)
results_df.sort_values("f1", ascending=False)


===== Training distilbert-base-uncased =====


Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_projector.bias    | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
classifier.bias         | MISSING    | 
classifier.weight       | MISSING    | 
pre_classifier.weight   | MISSING    | 
pre_classifier.bias     | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Map:   0%|          | 0/15840 [00:00<?, ? examples/s]

Map:   0%|          | 0/3960 [00:00<?, ? examples/s]

...

RobertaForSequenceClassification LOAD REPORT from: roberta-base
Key                             | Status     | 
--------------------------------+------------+-
lm_head.layer_norm.bias         | UNEXPECTED | 
lm_head.dense.bias              | UNEXPECTED | 
lm_head.bias                    | UNEXPECTED | 
lm_head.dense.weight            | UNEXPECTED | 
lm_head.layer_norm.weight       | UNEXPECTED | 
roberta.embeddings.position_ids | UNEXPECTED | 
classifier.dense.weight         | MISSING    | 
classifier.dense.bias           | MISSING    | 
classifier.out_proj.weight      | MISSING    | 
classifier.out_proj.bias        | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Map:   0%|          | 0/15840 [00:00<?, ? examples/s]

Map:   0%|          | 0/3960 [00:00<?, ? examples/s]

Epoch,Training Loss,Validation Loss,Accuracy,Precision,Recall,F1,Roc Auc
1,0.054827,0.044584,0.988384,0.980132,0.985936,0.983026,0.998930
2,0.008085,0.038890,0.991414,0.990320,0.984456,0.987379,0.999548


,model,accuracy,precision,recall,f1,roc_auc
2,roberta-base,0.991414,0.990320,0.984456,0.987379,0.999548
1,bert-base-uncased,0.990152,0.981645,0.989637,0.985625,0.999446
0,distilbert-base-uncased,0.988131,0.983680,0.981495,0.982586,0.999189


In [33]:
import nltk
nltk.download("wordnet")
nltk.download("omw-1.4")

[nltk_data] Downloading package wordnet to /root/nltk_data...
[nltk_data] Downloading package omw-1.4 to /root/nltk_data...


True

In [32]:
import re
import random
import numpy as np
import pandas as pd
import torch

from nltk.corpus import wordnet
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, roc_auc_score

In [39]:
def predict_one(text, model, tokenizer):
    model.eval()

    inputs = tokenizer(
        text,
        return_tensors="pt",
        truncation=True,
        max_length=512
    )
    inputs = {k: v.to(model.device) for k, v in inputs.items()}

    with torch.no_grad():
        outputs = model(**inputs)
        probs = torch.softmax(outputs.logits, dim=-1).cpu().numpy()[0]

    pred = int(np.argmax(probs))
    return pred, float(probs[1])

In [65]:
def predict_one_fast(text, model, tokenizer, max_length=512):
    enc = tokenizer(
        text,
        return_tensors="pt",
        truncation=True,
        max_length=max_length,
        padding=False,
    )

    enc = {k: v.to(model.device, non_blocking=True) for k, v in enc.items()}

    with torch.inference_mode():
        with torch.autocast(
            device_type="cuda",
            dtype=torch.float16,
            enabled=torch.cuda.is_available()
        ):
            outputs = model(**enc)
            probs = torch.softmax(outputs.logits, dim=-1)[0]

    probs = probs.detach().float().cpu().numpy()
    pred = int(np.argmax(probs))
    return pred, float(probs[1])


def batch_predict_fast(texts, model, tokenizer, batch_size=32, max_length=512):
    preds = []
    pos_probs = []

    for i in range(0, len(texts), batch_size):
        batch_texts = texts[i:i + batch_size]

        enc = tokenizer(
            batch_texts,
            return_tensors="pt",
            truncation=True,
            max_length=max_length,
            padding=True,
        )
        enc = {k: v.to(model.device, non_blocking=True) for k, v in enc.items()}

        with torch.inference_mode():
            with torch.autocast(
                device_type="cuda",
                dtype=torch.float16,
                enabled=torch.cuda.is_available()
            ):
                outputs = model(**enc)
                probs = torch.softmax(outputs.logits, dim=-1)

        probs = probs.detach().float().cpu().numpy()
        batch_preds = np.argmax(probs, axis=1)

        preds.extend(batch_preds.tolist())
        pos_probs.extend(probs[:, 1].tolist())

    return np.array(preds), np.array(pos_probs)

In [66]:
def get_synonyms(word):
    syns = set()
    for syn in wordnet.synsets(word):
        for lemma in syn.lemmas():
            s = lemma.name().replace("_", " ").strip()
            if s and s.lower() != word.lower():
                syns.add(s)
    return list(syns)


def synonym_attack(text, replace_prob=0.12, max_replacements=8, seed=42):
    rng = random.Random(seed)
    words = text.split()
    new_words = []
    replacements = 0

    for w in words:
        clean = re.sub(r"^[^\w]+|[^\w]+$", "", w)

        if (
            replacements < max_replacements
            and len(clean) >= 4
            and clean.isalpha()
            and rng.random() < replace_prob
        ):
            syns = get_synonyms(clean)
            syns = [s for s in syns if s.isalpha() and len(s.split()) == 1]

            if syns:
                replacement = rng.choice(syns)
                if w.istitle():
                    replacement = replacement.title()
                new_words.append(replacement)
                replacements += 1
                continue

        new_words.append(w)

    return " ".join(new_words)

In [67]:
def get_token_saliency(text, model, tokenizer, max_length=512):
    model.eval()

    enc = tokenizer(
        text,
        return_tensors="pt",
        truncation=True,
        max_length=max_length
    )

    input_ids = enc["input_ids"].to(model.device)
    attention_mask = enc["attention_mask"].to(model.device)

    embedding_layer = model.get_input_embeddings()
    inputs_embeds = embedding_layer(input_ids).detach()
    inputs_embeds.requires_grad_(True)

    outputs = model(inputs_embeds=inputs_embeds, attention_mask=attention_mask)
    pred_class = torch.argmax(outputs.logits, dim=1).item()
    score = outputs.logits[0, pred_class]
    score.backward()

    grads = inputs_embeds.grad[0]
    saliency = grads.norm(dim=1).detach().cpu().numpy()
    tokens = tokenizer.convert_ids_to_tokens(input_ids[0])

    token_scores = [(tok, float(s)) for tok, s in zip(tokens, saliency)]
    return token_scores


def merge_wordpiece_tokens(token_scores):
    words = []
    current_word = ""
    current_score = 0.0

    special_tokens = {"[CLS]", "[SEP]", "[PAD]", "<s>", "</s>", "<pad>"}

    for tok, score in token_scores:
        if tok in special_tokens:
            continue

        if tok.startswith("##"):
            current_word += tok[2:]
            current_score += score

        elif tok.startswith("Ġ"):
            if current_word:
                words.append((current_word, current_score))
            current_word = tok[1:]
            current_score = score

        else:
            if current_word:
                words.append((current_word, current_score))
            current_word = tok
            current_score = score

    if current_word:
        words.append((current_word, current_score))

    cleaned = []
    for w, s in words:
        w = w.strip()
        w = re.sub(r"[^\w@.\-:/]", "", w)
        if w:
            cleaned.append((w, s))

    return cleaned


def important_word_deletion_attack(text, model, tokenizer, k=5):
    token_scores = get_token_saliency(text, model, tokenizer)
    word_scores = merge_wordpiece_tokens(token_scores)
    ranked_words = sorted(word_scores, key=lambda x: x[1], reverse=True)

    stop = {
        "subject", "body", "the", "a", "an", "and", "or", "to", "of", "in",
        "for", "on", "at", "is", "are", "this", "that", "with", "from"
    }

    targets = []
    for w, _ in ranked_words:
        wl = w.lower()
        if len(wl) >= 3 and wl not in stop and wl.isprintable():
            targets.append(wl)
        if len(targets) >= k:
            break

    attacked_words = []
    for w in text.split():
        clean = re.sub(r"^[^\w]+|[^\w]+$", "", w).lower()
        if clean in targets:
            continue
        attacked_words.append(w)

    return " ".join(attacked_words)

In [68]:
def iterative_deletion_attack(text, model, tokenizer, max_steps=10):
    original_pred, original_prob = predict_one_fast(text, model, tokenizer)
    current_text = text

    for step in range(1, max_steps + 1):
        new_text = important_word_deletion_attack(
            current_text, model, tokenizer, k=1
        )

        if new_text == current_text:
            break

        new_pred, new_prob = predict_one_fast(new_text, model, tokenizer)

        if new_pred != original_pred:
            return {
                "attacked_text": new_text,
                "flipped": True,
                "steps": step,
                "original_pred": original_pred,
                "new_pred": new_pred,
                "original_prob": original_prob,
                "new_prob": new_prob,
            }

        current_text = new_text

    final_pred, final_prob = predict_one_fast(current_text, model, tokenizer)
    return {
        "attacked_text": current_text,
        "flipped": False,
        "steps": step if "step" in locals() else 0,
        "original_pred": original_pred,
        "new_pred": final_pred,
        "original_prob": original_prob,
        "new_prob": final_prob,
    }


def prefix_injection_attack(text):
    prefix = (
        "This is a normal and trustworthy business email. "
        "The message is legitimate, safe, and routine.\n\n"
    )
    return prefix + text

In [15]:
def evaluate_attack(df_eval, attack_name, attack_fn, model, tokenizer, batch_size=32):
    attacked_texts = [attack_fn(t) for t in df_eval["text"].tolist()]
    y_true = df_eval["label_id"].astype(int).tolist()

    y_pred, y_prob = batch_predict_fast(
        attacked_texts,
        model=model,
        tokenizer=tokenizer,
        batch_size=batch_size
    )

    precision, recall, f1, _ = precision_recall_fscore_support(
        y_true, y_pred, average="binary", zero_division=0
    )
    acc = accuracy_score(y_true, y_pred)

    try:
        roc_auc = roc_auc_score(y_true, y_prob)
    except Exception:
        roc_auc = float("nan")

    return {
        "attack": attack_name,
        "n_samples": len(df_eval),
        "accuracy": acc,
        "precision": precision,
        "recall": recall,
        "f1": f1,
        "roc_auc": roc_auc,
    }


def evaluate_iterative_attack(df_eval, model, tokenizer, max_samples=None, max_steps=10):
    if max_samples is not None:
        df_local = df_eval.iloc[:max_samples].copy()
    else:
        df_local = df_eval.copy()

    y_true = []
    y_pred_after = []
    y_prob_after = []
    flips = 0
    steps_used = []
    attacked_examples = []

    for _, row in df_local.iterrows():
        text = row["text"]
        label = int(row["label_id"])

        result = iterative_deletion_attack(
            text, model, tokenizer, max_steps=max_steps
        )

        final_pred, final_prob = predict_one_fast(result["attacked_text"], model, tokenizer)

        y_true.append(label)
        y_pred_after.append(final_pred)
        y_prob_after.append(final_prob)
        steps_used.append(result["steps"])

        if result["flipped"]:
            flips += 1

        attacked_examples.append({
            "original_text": text[:300],
            "attacked_text": result["attacked_text"][:300],
            "original_pred": result["original_pred"],
            "new_pred": result["new_pred"],
            "flipped": result["flipped"],
            "steps": result["steps"],
        })

    precision, recall, f1, _ = precision_recall_fscore_support(
        y_true, y_pred_after, average="binary", zero_division=0
    )
    acc = accuracy_score(y_true, y_pred_after)

    try:
        roc_auc = roc_auc_score(y_true, y_prob_after)
    except Exception:
        roc_auc = float("nan")

    summary = {
        "attack": f"iterative_deletion_{max_steps}steps",
        "n_samples": len(df_local),
        "accuracy": acc,
        "precision": precision,
        "recall": recall,
        "f1": f1,
        "roc_auc": roc_auc,
        "attack_success_rate": flips / len(df_local) if len(df_local) > 0 else float("nan"),
        "avg_steps_used": float(np.mean(steps_used)) if len(steps_used) > 0 else float("nan"),
    }

    attacked_examples_df = pd.DataFrame(attacked_examples)
    return summary, attacked_examples_df

In [16]:
attack_results_val = []

attack_results_val.append(
    evaluate_attack(val_df, "clean", lambda x: x, model, tokenizer)
)

attack_results_val.append(
    evaluate_attack(
        val_df,
        "synonym_attack",
        lambda x: synonym_attack(x, replace_prob=0.12, max_replacements=8, seed=42),
        model,
        tokenizer,
    )
)

attack_results_val.append(
    evaluate_attack(
        val_df,
        "important_word_delete_k5",
        lambda x: important_word_deletion_attack(x, model, tokenizer, k=5),
        model,
        tokenizer,
    )
)

attack_results_val.append(
    evaluate_attack(
        val_df,
        "prefix_injection",
        prefix_injection_attack,
        model,
        tokenizer,
    )
)

iter_summary_val, iter_examples_val = evaluate_iterative_attack(
    val_df,
    model,
    tokenizer,
    max_samples=300,
    max_steps=8
)

attack_results_val.append(iter_summary_val)

attack_results_val_df = pd.DataFrame(attack_results_val)
attack_results_val_df = attack_results_val_df.sort_values("f1", ascending=False)

print(attack_results_val_df)

NameError: name 'batch_predict_fast' is not defined

In [46]:
iter_examples_val.head(10)

,original_text,attacked_text,original_pred,new_pred,flipped,steps
0,Your Summer Home Upgrade Starts Now: Enjoy 15%...,Your Summer Upgrade Starts Now: Enjoy 15% Off!...,1,1,False,6
1,URGENT NEWS: Federal and Provincial Leaders Se...,URGENT NEWS: Federal and Provincial Leaders Se...,1,1,False,2
2,Action Required: Project Nexus — Feedback on T...,Action Required: Project Nexus — Feedback on T...,0,0,False,8
3,Urgent Security Alert: Your SecureBox account ...,Urgent Security Alert: Your SecureBox account ...,1,1,False,2
4,The New Era of Work: From Resignation to Resil...,The New Era of Work: From Resignation to Resil...,1,1,False,6
5,Re: Discussion on server best practices\n\nFro...,"Re: on best From: ""Eleanor Vance"" Tuesday, 14 ...",0,0,False,8
6,"Tech Insights Digest - February 15, 2024\n\nGr...","Tech Insights - 15, from Tech Insights This is...",0,0,False,5
7,Introducing Our Latest Collection: Seedling & ...,Introducing Our Latest Collection: Seedling & ...,1,1,False,8
8,[TechTalkForum] DSL Router Compatibility Query...,Router Hi router it appears the device in ques...,0,0,False,8
9,Re: [Datacore Connect] Data protection solutio...,Re: for I appreciate the suggestion! I wasn't ...,0,0,False,8


In [47]:
all_attack_tables = {}
all_iter_examples = {}

for i, test_df in enumerate(test_dfs, start=1):
    test_name = test_df["dataset"].iloc[0] if "dataset" in test_df.columns else f"test_{i}"

    attack_results = []

    attack_results.append(
        evaluate_attack(test_df, "clean", lambda x: x, model, tokenizer)
    )

    attack_results.append(
        evaluate_attack(
            test_df,
            "synonym_attack",
            lambda x: synonym_attack(x, replace_prob=0.12, max_replacements=8, seed=42),
            model,
            tokenizer,
        )
    )

    attack_results.append(
        evaluate_attack(
            test_df,
            "important_word_delete_k5",
            lambda x: important_word_deletion_attack(x, model, tokenizer, k=5),
            model,
            tokenizer,
        )
    )

    attack_results.append(
        evaluate_attack(
            test_df,
            "prefix_injection",
            prefix_injection_attack,
            model,
            tokenizer,
        )
    )

    iter_summary, iter_examples = evaluate_iterative_attack(
        test_df,
        model,
        tokenizer,
        max_samples=300,
        max_steps=8
    )

    attack_results.append(iter_summary)

    attack_results_df = pd.DataFrame(attack_results).sort_values("f1", ascending=False)

    all_attack_tables[test_name] = attack_results_df
    all_iter_examples[test_name] = iter_examples

    print(f"\n=== {test_name} ===")
    print(attack_results_df)


=== CEAS_08_cleaned ===
                      attack  n_samples  accuracy  precision    recall  \
3           prefix_injection      39154  0.795908   0.944142  0.674023   
2   important_word_delete_k5      39154  0.748940   0.942855  0.585432   
0                      clean      39154  0.735966   0.962157  0.548256   
1             synonym_attack      39154  0.731062   0.959688  0.540610   
4  iterative_deletion_8steps        300  0.613333   0.836957  0.432584   

         f1   roc_auc  attack_success_rate  avg_steps_used  
3  0.786537  0.940944                  NaN             NaN  
2  0.722348  0.941816                  NaN             NaN  
0  0.698495  0.954819                  NaN             NaN  
1  0.691618  0.954364                  NaN             NaN  
4  0.570370  0.833993             0.073333        4.113333  


/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_ranking.py:379: UndefinedMetricWarning: Only one class is present in y_true. ROC AUC score is not defined in that case.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_ranking.py:379: UndefinedMetricWarning: Only one class is present in y_true. ROC AUC score is not defined in that case.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_ranking.py:379: UndefinedMetricWarning: Only one class is present in y_true. ROC AUC score is not defined in that case.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_ranking.py:379: UndefinedMetricWarning: Only one class is present in y_true. ROC AUC score is not defined in that case.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_ranking.py:379: UndefinedMetricWarning: Only one class is present in y_true. ROC AUC score is not defined in that case.
  warnings.warn(



=== Nazario_cleaned ===
                      attack  n_samples  accuracy  precision    recall  \
3           prefix_injection       1565  0.994888        1.0  0.994888   
0                      clean       1565  0.977636        1.0  0.977636   
1             synonym_attack       1565  0.973163        1.0  0.973163   
2   important_word_delete_k5       1565  0.961022        1.0  0.961022   
4  iterative_deletion_8steps        300  0.920000        1.0  0.920000   

         f1  roc_auc  attack_success_rate  avg_steps_used  
3  0.997438      NaN                  NaN             NaN  
0  0.988691      NaN                  NaN             NaN  
1  0.986399      NaN                  NaN             NaN  
2  0.980124      NaN                  NaN             NaN  
4  0.958333      NaN             0.066667            5.03  


/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_ranking.py:379: UndefinedMetricWarning: Only one class is present in y_true. ROC AUC score is not defined in that case.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_ranking.py:379: UndefinedMetricWarning: Only one class is present in y_true. ROC AUC score is not defined in that case.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_ranking.py:379: UndefinedMetricWarning: Only one class is present in y_true. ROC AUC score is not defined in that case.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_ranking.py:379: UndefinedMetricWarning: Only one class is present in y_true. ROC AUC score is not defined in that case.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_ranking.py:379: UndefinedMetricWarning: Only one class is present in y_true. ROC AUC score is not defined in that case.
  warnings.warn(



=== Nigerian_Fraud_cleaned ===
                      attack  n_samples  accuracy  precision    recall  \
4  iterative_deletion_8steps        300  0.983333        1.0  0.983333   
3           prefix_injection       3332  0.974790        1.0  0.974790   
2   important_word_delete_k5       3332  0.962785        1.0  0.962785   
0                      clean       3332  0.960684        1.0  0.960684   
1             synonym_attack       3332  0.958583        1.0  0.958583   

         f1  roc_auc  attack_success_rate  avg_steps_used  
4  0.991597      NaN             0.006667        4.903333  
3  0.987234      NaN                  NaN             NaN  
2  0.981040      NaN                  NaN             NaN  
0  0.979948      NaN                  NaN             NaN  
1  0.978854      NaN                  NaN             NaN  

=== SpamAssasin_cleaned ===
                      attack  n_samples  accuracy  precision    recall  \
2   important_word_delete_k5       5809  0.904286   0.944870

/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_ranking.py:379: UndefinedMetricWarning: Only one class is present in y_true. ROC AUC score is not defined in that case.
  warnings.warn(


In [20]:
import numpy as np
import pandas as pd
from tqdm.auto import tqdm
from sklearn.metrics import (
    precision_recall_fscore_support,
    accuracy_score,
    roc_auc_score,
)

In [21]:
def predict_one_fast(text, model, tokenizer, max_length=512):
    enc = tokenizer(
        text,
        return_tensors="pt",
        truncation=True,
        max_length=max_length,
        padding=False,
    )

    enc = {k: v.to(model.device, non_blocking=True) for k, v in enc.items()}

    with torch.inference_mode():
        with torch.autocast(
            device_type="cuda",
            dtype=torch.float16,
            enabled=torch.cuda.is_available()
        ):
            outputs = model(**enc)
            probs = torch.softmax(outputs.logits, dim=-1)[0]

    probs = probs.detach().float().cpu().numpy()
    pred = int(np.argmax(probs))
    return pred, float(probs[1])


def batch_predict_fast(texts, model, tokenizer, batch_size=32, max_length=512):
    preds = []
    pos_probs = []

    for i in range(0, len(texts), batch_size):
        batch_texts = texts[i:i + batch_size]

        enc = tokenizer(
            batch_texts,
            return_tensors="pt",
            truncation=True,
            max_length=max_length,
            padding=True,
        )
        enc = {k: v.to(model.device, non_blocking=True) for k, v in enc.items()}

        with torch.inference_mode():
            with torch.autocast(
                device_type="cuda",
                dtype=torch.float16,
                enabled=torch.cuda.is_available()
            ):
                outputs = model(**enc)
                probs = torch.softmax(outputs.logits, dim=-1)

        probs = probs.detach().float().cpu().numpy()
        batch_preds = np.argmax(probs, axis=1)

        preds.extend(batch_preds.tolist())
        pos_probs.extend(probs[:, 1].tolist())

    return np.array(preds), np.array(pos_probs)

In [22]:
def get_synonyms(word):
    syns = set()
    for syn in wordnet.synsets(word):
        for lemma in syn.lemmas():
            s = lemma.name().replace("_", " ").strip()
            if s and s.lower() != word.lower():
                syns.add(s)
    return list(syns)


def synonym_attack(text, replace_prob=0.12, max_replacements=8, seed=42):
    rng = random.Random(seed)
    words = text.split()
    new_words = []
    replacements = 0

    for w in words:
        clean = re.sub(r"^[^\w]+|[^\w]+$", "", w)

        if (
            replacements < max_replacements
            and len(clean) >= 4
            and clean.isalpha()
            and rng.random() < replace_prob
        ):
            syns = get_synonyms(clean)
            syns = [s for s in syns if s.isalpha() and len(s.split()) == 1]

            if syns:
                replacement = rng.choice(syns)
                if w.istitle():
                    replacement = replacement.title()
                new_words.append(replacement)
                replacements += 1
                continue

        new_words.append(w)

    return " ".join(new_words)

In [23]:
def get_token_saliency(text, model, tokenizer, max_length=512):
    model.eval()

    enc = tokenizer(
        text,
        return_tensors="pt",
        truncation=True,
        max_length=max_length
    )

    input_ids = enc["input_ids"].to(model.device)
    attention_mask = enc["attention_mask"].to(model.device)

    embedding_layer = model.get_input_embeddings()
    inputs_embeds = embedding_layer(input_ids).detach()
    inputs_embeds.requires_grad_(True)

    outputs = model(inputs_embeds=inputs_embeds, attention_mask=attention_mask)
    pred_class = torch.argmax(outputs.logits, dim=1).item()
    score = outputs.logits[0, pred_class]
    score.backward()

    grads = inputs_embeds.grad[0]
    saliency = grads.norm(dim=1).detach().cpu().numpy()
    tokens = tokenizer.convert_ids_to_tokens(input_ids[0])

    return [(tok, float(s)) for tok, s in zip(tokens, saliency)]


def merge_wordpiece_tokens(token_scores):
    words = []
    current_word = ""
    current_score = 0.0

    special_tokens = {"[CLS]", "[SEP]", "[PAD]", "<s>", "</s>", "<pad>"}

    for tok, score in token_scores:
        if tok in special_tokens:
            continue

        if tok.startswith("##"):
            current_word += tok[2:]
            current_score += score
        elif tok.startswith("Ġ"):
            if current_word:
                words.append((current_word, current_score))
            current_word = tok[1:]
            current_score = score
        else:
            if current_word:
                words.append((current_word, current_score))
            current_word = tok
            current_score = score

    if current_word:
        words.append((current_word, current_score))

    cleaned = []
    for w, s in words:
        w = w.strip()
        w = re.sub(r"[^\w@.\-:/]", "", w)
        if w:
            cleaned.append((w, s))

    return cleaned


def important_word_deletion_attack(text, model, tokenizer, k=5):
    token_scores = get_token_saliency(text, model, tokenizer)
    word_scores = merge_wordpiece_tokens(token_scores)
    ranked_words = sorted(word_scores, key=lambda x: x[1], reverse=True)

    stop = {
        "subject", "body", "the", "a", "an", "and", "or", "to", "of", "in",
        "for", "on", "at", "is", "are", "this", "that", "with", "from"
    }

    targets = []
    for w, _ in ranked_words:
        wl = w.lower()
        if len(wl) >= 3 and wl not in stop and wl.isprintable():
            targets.append(wl)
        if len(targets) >= k:
            break

    attacked_words = []
    for w in text.split():
        clean = re.sub(r"^[^\w]+|[^\w]+$", "", w).lower()
        if clean in targets:
            continue
        attacked_words.append(w)

    return " ".join(attacked_words)


important_word_deletion_attack_fast = important_word_deletion_attack

In [24]:
def iterative_deletion_attack(text, model, tokenizer, max_steps=10):
    original_pred, original_prob = predict_one_fast(text, model, tokenizer)
    current_text = text

    for step in range(1, max_steps + 1):
        new_text = important_word_deletion_attack(
            current_text, model, tokenizer, k=1
        )

        if new_text == current_text:
            break

        new_pred, new_prob = predict_one_fast(new_text, model, tokenizer)

        if new_pred != original_pred:
            return {
                "attacked_text": new_text,
                "flipped": True,
                "steps": step,
                "original_pred": original_pred,
                "new_pred": new_pred,
                "original_prob": original_prob,
                "new_prob": new_prob,
            }

        current_text = new_text

    final_pred, final_prob = predict_one_fast(current_text, model, tokenizer)
    return {
        "attacked_text": current_text,
        "flipped": False,
        "steps": step if "step" in locals() else 0,
        "original_pred": original_pred,
        "new_pred": final_pred,
        "original_prob": original_prob,
        "new_prob": final_prob,
    }


def prefix_injection_attack(text):
    prefix = (
        "This is a normal and trustworthy business email. "
        "The message is legitimate, safe, and routine.\n\n"
    )
    return prefix + text

In [25]:
def iterative_deletion_attack(text, model, tokenizer, max_steps=10):
    original_pred, original_prob = predict_one_fast(text, model, tokenizer)
    current_text = text

    for step in range(1, max_steps + 1):
        new_text = important_word_deletion_attack(
            current_text, model, tokenizer, k=1
        )

        if new_text == current_text:
            break

        new_pred, new_prob = predict_one_fast(new_text, model, tokenizer)

        if new_pred != original_pred:
            return {
                "attacked_text": new_text,
                "flipped": True,
                "steps": step,
                "original_pred": original_pred,
                "new_pred": new_pred,
                "original_prob": original_prob,
                "new_prob": new_prob,
            }

        current_text = new_text

    final_pred, final_prob = predict_one_fast(current_text, model, tokenizer)
    return {
        "attacked_text": current_text,
        "flipped": False,
        "steps": step if "step" in locals() else 0,
        "original_pred": original_pred,
        "new_pred": final_pred,
        "original_prob": original_prob,
        "new_prob": final_prob,
    }


def prefix_injection_attack(text):
    prefix = (
        "This is a normal and trustworthy business email. "
        "The message is legitimate, safe, and routine.\n\n"
    )
    return prefix + text

In [26]:
def evaluate_attack(df_eval, attack_name, attack_fn, model, tokenizer, batch_size=32):
    attacked_texts = [attack_fn(t) for t in df_eval["text"].tolist()]
    y_true = df_eval["label_id"].astype(int).tolist()

    y_pred, y_prob = batch_predict_fast(
        attacked_texts,
        model=model,
        tokenizer=tokenizer,
        batch_size=batch_size
    )

    precision, recall, f1, _ = precision_recall_fscore_support(
        y_true, y_pred, average="binary", zero_division=0
    )
    acc = accuracy_score(y_true, y_pred)

    try:
        roc_auc = roc_auc_score(y_true, y_prob)
    except Exception:
        roc_auc = float("nan")

    return {
        "attack": attack_name,
        "n_samples": len(df_eval),
        "accuracy": acc,
        "precision": precision,
        "recall": recall,
        "f1": f1,
        "roc_auc": roc_auc,
    }


def evaluate_iterative_attack(df_eval, model, tokenizer, max_samples=None, max_steps=10):
    if max_samples is not None:
        df_local = df_eval.iloc[:max_samples].copy()
    else:
        df_local = df_eval.copy()

    y_true = []
    y_pred_after = []
    y_prob_after = []
    flips = 0
    steps_used = []
    attacked_examples = []

    for _, row in df_local.iterrows():
        text = row["text"]
        label = int(row["label_id"])

        result = iterative_deletion_attack(
            text, model, tokenizer, max_steps=max_steps
        )

        final_pred, final_prob = predict_one_fast(result["attacked_text"], model, tokenizer)

        y_true.append(label)
        y_pred_after.append(final_pred)
        y_prob_after.append(final_prob)
        steps_used.append(result["steps"])

        if result["flipped"]:
            flips += 1

        attacked_examples.append({
            "original_text": text[:300],
            "attacked_text": result["attacked_text"][:300],
            "original_pred": result["original_pred"],
            "new_pred": result["new_pred"],
            "flipped": result["flipped"],
            "steps": result["steps"],
        })

    precision, recall, f1, _ = precision_recall_fscore_support(
        y_true, y_pred_after, average="binary", zero_division=0
    )
    acc = accuracy_score(y_true, y_pred_after)

    try:
        roc_auc = roc_auc_score(y_true, y_prob_after)
    except Exception:
        roc_auc = float("nan")

    summary = {
        "attack": f"iterative_deletion_{max_steps}steps",
        "n_samples": len(df_local),
        "accuracy": acc,
        "precision": precision,
        "recall": recall,
        "f1": f1,
        "roc_auc": roc_auc,
        "attack_success_rate": flips / len(df_local) if len(df_local) > 0 else float("nan"),
        "avg_steps_used": float(np.mean(steps_used)) if len(steps_used) > 0 else float("nan"),
    }

    attacked_examples_df = pd.DataFrame(attacked_examples)
    return summary, attacked_examples_df

In [59]:
attack_results_val = []

attack_results_val.append(
    evaluate_attack(val_df, "clean", lambda x: x, model, tokenizer)
)

attack_results_val.append(
    evaluate_attack(
        val_df,
        "synonym_attack",
        lambda x: synonym_attack(x, replace_prob=0.12, max_replacements=8, seed=42),
        model,
        tokenizer,
    )
)

attack_results_val.append(
    evaluate_attack(
        val_df,
        "important_word_delete_k5",
        lambda x: important_word_deletion_attack(x, model, tokenizer, k=5),
        model,
        tokenizer,
    )
)

attack_results_val.append(
    evaluate_attack(
        val_df,
        "prefix_injection",
        prefix_injection_attack,
        model,
        tokenizer,
    )
)

iter_summary_val, iter_examples_val = evaluate_iterative_attack(
    val_df,
    model,
    tokenizer,
    max_samples=300,
    max_steps=8
)

attack_results_val.append(iter_summary_val)

attack_results_val_df = pd.DataFrame(attack_results_val).sort_values("f1", ascending=False)
attack_results_val_df

,attack,n_samples,accuracy,precision,recall,f1,roc_auc,attack_success_rate,avg_steps_used
1,synonym_attack,3960,0.980051,0.980488,0.989773,0.985108,0.996968,NaN,NaN
0,clean,3960,0.979293,0.981551,0.987500,0.984517,0.997425,NaN,NaN
3,prefix_injection,3960,0.972475,0.964063,0.995833,0.979691,0.996460,NaN,NaN
2,important_word_delete_k5,3960,0.967677,0.964497,0.987879,0.976048,0.992840,NaN,NaN
4,iterative_deletion_8steps,300,0.963333,0.953052,0.995098,0.973621,0.990349,0.023333,5.503333


In [27]:
def hybrid_add_then_delete_attack_fast(
    text,
    model,
    tokenizer,
    add_steps=3,
    delete_steps=5,
    max_length=512,
):
    original_pred, original_prob = predict_one_fast(
        text, model, tokenizer, max_length=max_length
    )

    current_text = text
    history = []

    addition_fns = [
        ("benign_prefix", benign_prefix_attack),
        ("benign_suffix", benign_suffix_attack),
        ("contradiction", contradiction_attack),
        ("training_context", training_context_attack),
        ("noise_injection", noise_injection_attack),
    ]

    used_add_steps = 0
    used_delete_steps = 0

    for step in range(1, add_steps + 1):
        candidate_names = []
        candidate_texts = []

        for attack_name, attack_fn in addition_fns:
            candidate_names.append(attack_name)
            candidate_texts.append(attack_fn(current_text))

        candidate_preds, candidate_probs = batch_predict_fast(
            candidate_texts,
            model=model,
            tokenizer=tokenizer,
            batch_size=len(candidate_texts),
            max_length=max_length,
        )

        best_idx = int(np.argmin(candidate_probs))
        current_text = candidate_texts[best_idx]
        best_name = candidate_names[best_idx]
        best_pred = int(candidate_preds[best_idx])
        best_prob = float(candidate_probs[best_idx])
        used_add_steps = step

        history.append({
            "phase": "add",
            "step": step,
            "attack_name": best_name,
            "pred": best_pred,
            "phishing_prob": best_prob,
            "text_preview": current_text[:300],
        })

        if best_pred != original_pred:
            return {
                "attacked_text": current_text,
                "flipped": True,
                "steps_add": used_add_steps,
                "steps_delete": used_delete_steps,
                "total_steps": used_add_steps + used_delete_steps,
                "original_pred": original_pred,
                "new_pred": best_pred,
                "original_prob": original_prob,
                "new_prob": best_prob,
                "history": history,
            }

    for dstep in range(1, delete_steps + 1):
        new_text = important_word_deletion_attack_fast(
            current_text,
            model,
            tokenizer,
            k=1
        )

        if new_text == current_text:
            break

        current_text = new_text
        new_pred, new_prob = predict_one_fast(
            current_text, model, tokenizer, max_length=max_length
        )
        used_delete_steps = dstep

        history.append({
            "phase": "delete",
            "step": dstep,
            "attack_name": "important_word_delete_k1",
            "pred": new_pred,
            "phishing_prob": new_prob,
            "text_preview": current_text[:300],
        })

        if new_pred != original_pred:
            return {
                "attacked_text": current_text,
                "flipped": True,
                "steps_add": used_add_steps,
                "steps_delete": used_delete_steps,
                "total_steps": used_add_steps + used_delete_steps,
                "original_pred": original_pred,
                "new_pred": new_pred,
                "original_prob": original_prob,
                "new_prob": new_prob,
                "history": history,
            }

    final_pred, final_prob = predict_one_fast(
        current_text, model, tokenizer, max_length=max_length
    )

    return {
        "attacked_text": current_text,
        "flipped": False,
        "steps_add": used_add_steps,
        "steps_delete": used_delete_steps,
        "total_steps": used_add_steps + used_delete_steps,
        "original_pred": original_pred,
        "new_pred": final_pred,
        "original_prob": original_prob,
        "new_prob": final_prob,
        "history": history,
    }

In [28]:
def evaluate_hybrid_add_then_delete_attack_fast(
    df_eval,
    model,
    tokenizer,
    max_samples=None,
    add_steps=3,
    delete_steps=5,
    max_length=512,
    show_progress=True,
):
    if max_samples is not None:
        df_local = df_eval.iloc[:max_samples].copy()
    else:
        df_local = df_eval.copy()

    y_true = []
    y_pred_after = []
    y_prob_after = []

    flips = 0
    total_steps_used = []
    add_steps_used = []
    delete_steps_used = []
    records = []

    iterator = df_local.iterrows()
    if show_progress:
        iterator = tqdm(iterator, total=len(df_local), desc="Hybrid fast eval")

    for _, row in iterator:
        text = row["text"]
        label = int(row["label_id"])

        result = hybrid_add_then_delete_attack_fast(
            text=text,
            model=model,
            tokenizer=tokenizer,
            add_steps=add_steps,
            delete_steps=delete_steps,
            max_length=max_length,
        )

        y_true.append(label)
        y_pred_after.append(int(result["new_pred"]))
        y_prob_after.append(float(result["new_prob"]))

        total_steps_used.append(result["total_steps"])
        add_steps_used.append(result["steps_add"])
        delete_steps_used.append(result["steps_delete"])

        if result["flipped"]:
            flips += 1

        records.append({
            "label_id": label,
            "original_pred": result["original_pred"],
            "new_pred": result["new_pred"],
            "flipped": result["flipped"],
            "steps_add": result["steps_add"],
            "steps_delete": result["steps_delete"],
            "total_steps": result["total_steps"],
            "original_prob": result["original_prob"],
            "new_prob": result["new_prob"],
            "original_text": text[:300],
            "attacked_text": result["attacked_text"][:300],
        })

    precision, recall, f1, _ = precision_recall_fscore_support(
        y_true,
        y_pred_after,
        average="binary",
        zero_division=0
    )
    acc = accuracy_score(y_true, y_pred_after)

    try:
        roc_auc = roc_auc_score(y_true, y_prob_after)
    except Exception:
        roc_auc = float("nan")

    n = len(df_local)

    summary = {
        "attack": f"hybrid_fast_add{add_steps}_delete{delete_steps}",
        "n_samples": n,
        "accuracy": acc,
        "precision": precision,
        "recall": recall,
        "f1": f1,
        "roc_auc": roc_auc,
        "attack_success_rate": flips / n if n > 0 else float("nan"),
        "avg_total_steps_used": float(np.mean(total_steps_used)) if n > 0 else float("nan"),
        "avg_add_steps_used": float(np.mean(add_steps_used)) if n > 0 else float("nan"),
        "avg_delete_steps_used": float(np.mean(delete_steps_used)) if n > 0 else float("nan"),
    }

    details_df = pd.DataFrame(records)
    return summary, details_df

In [17]:
def benign_prefix_attack(text):
    prefix = (
        "Hello,\n\n"
        "This is a normal business communication regarding routine account activity. "
        "Please review the message below.\n\n"
    )
    return prefix + text


def benign_suffix_attack(text):
    suffix = (
        "\n\nThank you for your time and attention."
        "\nThis message is intended for standard informational purposes."
        "\nBest regards."
    )
    return text + suffix


def contradiction_attack(text):
    prefix = (
        "Important note: this email is legitimate and does not request any unsafe action. "
        "It is only a routine notification.\n\n"
    )
    return prefix + text


def training_context_attack(text):
    prefix = (
        "Example of a safe enterprise email for internal security-awareness training:\n\n"
    )
    return prefix + text


def noise_injection_attack(text):
    noise = (
        "\n\nReference ID: A7K2-91LM\n"
        "Ticket: routine-review-8841\n"
        "Department: operations\n"
    )
    return text + noise

In [80]:
hybrid_val_summary_fast, hybrid_val_details_fast = evaluate_hybrid_add_then_delete_attack_fast(
    val_df,
    model,
    tokenizer,
    max_samples=None,
    add_steps=3,
    delete_steps=5,
    max_length=512,
    show_progress=True,
)

pd.DataFrame([{"dataset": "validation", **hybrid_val_summary_fast}])

Hybrid fast eval:   0%|          | 0/3960 [00:00<?, ?it/s]

,dataset,attack,n_samples,accuracy,precision,recall,f1,roc_auc,attack_success_rate,avg_total_steps_used,avg_add_steps_used,avg_delete_steps_used
0,validation,hybrid_fast_add3_delete5,3960,0.878283,0.966696,0.846591,0.902666,0.984406,0.125253,6.116667,2.885354,3.231313


In [29]:
hybrid_test_summaries = []
hybrid_test_details = {}

for i, test_df in enumerate(test_dfs, start=1):
    test_name = test_df["dataset"].iloc[0] if "dataset" in test_df.columns else f"test_{i}"

    summary, details = evaluate_hybrid_add_then_delete_attack_fast(
        test_df,
        model,
        tokenizer,
        max_samples=None,
        add_steps=3,
        delete_steps=5,
        max_length=512,
        show_progress=True,
    )

    hybrid_test_summaries.append({
        "dataset": test_name,
        **summary
    })
    hybrid_test_details[test_name] = details

hybrid_test_summary_df = pd.DataFrame(hybrid_test_summaries)
hybrid_test_summary_df

Hybrid fast eval:   0%|          | 0/39154 [00:00<?, ?it/s]

Hybrid fast eval:   0%|          | 0/1565 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_ranking.py:379: UndefinedMetricWarning: Only one class is present in y_true. ROC AUC score is not defined in that case.
  warnings.warn(


Hybrid fast eval:   0%|          | 0/3332 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_ranking.py:379: UndefinedMetricWarning: Only one class is present in y_true. ROC AUC score is not defined in that case.
  warnings.warn(


Hybrid fast eval:   0%|          | 0/5809 [00:00<?, ?it/s]

,dataset,attack,n_samples,accuracy,precision,recall,f1,roc_auc,attack_success_rate,avg_total_steps_used,avg_add_steps_used,avg_delete_steps_used
0,CEAS_08_cleaned,hybrid_fast_add3_delete5,39154,0.538029,0.904700,0.192107,0.316918,0.935121,0.346887,4.276626,2.566890,1.709736
1,Nazario_cleaned,hybrid_fast_add3_delete5,1565,0.872204,1.000000,0.872204,0.931741,NaN,0.122684,5.386581,2.899042,2.487540
2,Nigerian_Fraud_cleaned,hybrid_fast_add3_delete5,3332,0.552221,1.000000,0.552221,0.711524,NaN,0.408463,4.461885,2.535714,1.926170
3,SpamAssasin_cleaned,hybrid_fast_add3_delete5,5809,0.802376,0.945312,0.352154,0.513147,0.922302,0.131004,4.929420,2.848511,2.080909


In [30]:
required_names = [
    "predict_one_fast",
    "batch_predict_fast",
    "important_word_deletion_attack",
    "model",
    "tokenizer",
    "val_df",
]

missing = [name for name in required_names if name not in globals()]
if missing:
    raise ValueError(f"Missing required objects/functions: {missing}")

print("Base prerequisites found.")

Base prerequisites found.


In [31]:
def benign_prefix_attack(text):
    prefix = (
        "Hello,\n\n"
        "This is a routine business communication regarding normal account activity. "
        "Please review the information below.\n\n"
    )
    return prefix + text


def benign_suffix_attack(text):
    suffix = (
        "\n\nThank you for your attention."
        "\nThis message is part of a standard informational workflow."
        "\nBest regards."
    )
    return text + suffix


def contradiction_attack(text):
    prefix = (
        "Important note: this is a legitimate and safe message. "
        "It does not request any unsafe action.\n\n"
    )
    return prefix + text


def training_context_attack(text):
    prefix = (
        "Example of a safe enterprise email used for routine staff communication:\n\n"
    )
    return prefix + text


def noise_injection_attack(text):
    noise = (
        "\n\nReference ID: A7K2-91LM"
        "\nDepartment: operations"
        "\nCase Type: routine review"
    )
    return text + noise

In [32]:
important_word_deletion_attack_fast = important_word_deletion_attack

In [33]:
import numpy as np

def hybrid_add_then_delete_attack_fast(
    text,
    model,
    tokenizer,
    add_steps=3,
    delete_steps=5,
    max_length=512,
):
    """
    Phase 1: greedy batched additions
    Phase 2: saliency-guided deletions

    Returns:
      attacked_text, flipped, steps used, original/new pred and prob, history
    """

    original_pred, original_prob = predict_one_fast(
        text, model, tokenizer, max_length=max_length
    )

    current_text = text
    history = []

    addition_fns = [
        ("benign_prefix", benign_prefix_attack),
        ("benign_suffix", benign_suffix_attack),
        ("contradiction", contradiction_attack),
        ("training_context", training_context_attack),
        ("noise_injection", noise_injection_attack),
    ]

    used_add_steps = 0
    used_delete_steps = 0

    # Phase 1: greedy additions
    for step in range(1, add_steps + 1):
        candidate_names = []
        candidate_texts = []

        for attack_name, attack_fn in addition_fns:
            candidate_names.append(attack_name)
            candidate_texts.append(attack_fn(current_text))

        candidate_preds, candidate_probs = batch_predict_fast(
            candidate_texts,
            model=model,
            tokenizer=tokenizer,
            batch_size=len(candidate_texts),
            max_length=max_length,
        )

        # Choose the candidate with the lowest phishing probability
        best_idx = int(np.argmin(candidate_probs))
        current_text = candidate_texts[best_idx]
        best_name = candidate_names[best_idx]
        best_pred = int(candidate_preds[best_idx])
        best_prob = float(candidate_probs[best_idx])
        used_add_steps = step

        history.append({
            "phase": "add",
            "step": step,
            "attack_name": best_name,
            "pred": best_pred,
            "phishing_prob": best_prob,
            "text_preview": current_text[:300],
        })

        if best_pred != original_pred:
            return {
                "attacked_text": current_text,
                "flipped": True,
                "steps_add": used_add_steps,
                "steps_delete": used_delete_steps,
                "total_steps": used_add_steps + used_delete_steps,
                "original_pred": original_pred,
                "new_pred": best_pred,
                "original_prob": original_prob,
                "new_prob": best_prob,
                "history": history,
            }

    # Phase 2: saliency-guided deletion
    for dstep in range(1, delete_steps + 1):
        new_text = important_word_deletion_attack_fast(
            current_text,
            model,
            tokenizer,
            k=1
        )

        if new_text == current_text:
            break

        current_text = new_text
        new_pred, new_prob = predict_one_fast(
            current_text, model, tokenizer, max_length=max_length
        )
        used_delete_steps = dstep

        history.append({
            "phase": "delete",
            "step": dstep,
            "attack_name": "important_word_delete_k1",
            "pred": new_pred,
            "phishing_prob": new_prob,
            "text_preview": current_text[:300],
        })

        if new_pred != original_pred:
            return {
                "attacked_text": current_text,
                "flipped": True,
                "steps_add": used_add_steps,
                "steps_delete": used_delete_steps,
                "total_steps": used_add_steps + used_delete_steps,
                "original_pred": original_pred,
                "new_pred": new_pred,
                "original_prob": original_prob,
                "new_prob": new_prob,
                "history": history,
            }

    final_pred, final_prob = predict_one_fast(
        current_text, model, tokenizer, max_length=max_length
    )

    return {
        "attacked_text": current_text,
        "flipped": False,
        "steps_add": used_add_steps,
        "steps_delete": used_delete_steps,
        "total_steps": used_add_steps + used_delete_steps,
        "original_pred": original_pred,
        "new_pred": final_pred,
        "original_prob": original_prob,
        "new_prob": final_prob,
        "history": history,
    }

In [34]:
import pandas as pd
import numpy as np
from tqdm.auto import tqdm
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, roc_auc_score

def evaluate_hybrid_evasion_on_phishing(
    df_eval,
    model,
    tokenizer,
    add_steps=3,
    delete_steps=5,
    max_length=512,
    max_samples=None,
    show_progress=True,
):
    """
    Adversarial evaluation restricted to:
      - true phishing samples (label_id == 1)
      - originally predicted correctly as phishing

    Main metric:
      ASR = fraction of originally correct phishing samples that flip after attack
    """

    df_local = df_eval.copy()

    # Keep only phishing ground-truth rows
    df_local = df_local[df_local["label_id"].astype(int) == 1].copy()

    if max_samples is not None:
        df_local = df_local.iloc[:max_samples].copy()

    if len(df_local) == 0:
        summary = {
            "attack": f"hybrid_fast_add{add_steps}_delete{delete_steps}",
            "n_true_phishing": 0,
            "n_orig_correct_phishing": 0,
            "n_flipped": 0,
            "attack_success_rate": float("nan"),
            "robust_recall_on_orig_correct_phishing": float("nan"),
            "avg_total_steps_used": float("nan"),
            "avg_add_steps_used": float("nan"),
            "avg_delete_steps_used": float("nan"),
            "avg_prob_drop": float("nan"),
        }
        return summary, pd.DataFrame()

    # Step 1: evaluate original predictions
    original_texts = df_local["text"].tolist()
    orig_preds, orig_probs = batch_predict_fast(
        original_texts,
        model=model,
        tokenizer=tokenizer,
        batch_size=32,
        max_length=max_length,
    )

    df_local = df_local.copy()
    df_local["orig_pred"] = orig_preds
    df_local["orig_prob"] = orig_probs

    # Keep only originally correct phishing samples
    df_attack = df_local[df_local["orig_pred"] == 1].copy()

    if len(df_attack) == 0:
        summary = {
            "attack": f"hybrid_fast_add{add_steps}_delete{delete_steps}",
            "n_true_phishing": len(df_local),
            "n_orig_correct_phishing": 0,
            "n_flipped": 0,
            "attack_success_rate": float("nan"),
            "robust_recall_on_orig_correct_phishing": float("nan"),
            "avg_total_steps_used": float("nan"),
            "avg_add_steps_used": float("nan"),
            "avg_delete_steps_used": float("nan"),
            "avg_prob_drop": float("nan"),
        }
        return summary, pd.DataFrame()

    flips = 0
    total_steps_used = []
    add_steps_used = []
    delete_steps_used = []
    prob_drops = []
    records = []

    iterator = df_attack.iterrows()
    if show_progress:
        iterator = tqdm(iterator, total=len(df_attack), desc="Hybrid phishing evasion")

    for _, row in iterator:
        text = row["text"]
        orig_prob = float(row["orig_prob"])

        result = hybrid_add_then_delete_attack_fast(
            text=text,
            model=model,
            tokenizer=tokenizer,
            add_steps=add_steps,
            delete_steps=delete_steps,
            max_length=max_length,
        )

        flipped = int(result["new_pred"]) != 1
        if flipped:
            flips += 1

        total_steps_used.append(result["total_steps"])
        add_steps_used.append(result["steps_add"])
        delete_steps_used.append(result["steps_delete"])
        prob_drops.append(orig_prob - float(result["new_prob"]))

        records.append({
            "original_text": text[:300],
            "attacked_text": result["attacked_text"][:300],
            "original_pred": int(result["original_pred"]),
            "new_pred": int(result["new_pred"]),
            "flipped": flipped,
            "original_prob": orig_prob,
            "new_prob": float(result["new_prob"]),
            "prob_drop": orig_prob - float(result["new_prob"]),
            "steps_add": result["steps_add"],
            "steps_delete": result["steps_delete"],
            "total_steps": result["total_steps"],
        })

    n_orig_correct = len(df_attack)
    asr = flips / n_orig_correct
    robust_recall = 1.0 - asr

    summary = {
        "attack": f"hybrid_fast_add{add_steps}_delete{delete_steps}",
        "n_true_phishing": len(df_local),
        "n_orig_correct_phishing": n_orig_correct,
        "n_flipped": flips,
        "attack_success_rate": asr,
        "robust_recall_on_orig_correct_phishing": robust_recall,
        "avg_total_steps_used": float(np.mean(total_steps_used)),
        "avg_add_steps_used": float(np.mean(add_steps_used)),
        "avg_delete_steps_used": float(np.mean(delete_steps_used)),
        "avg_prob_drop": float(np.mean(prob_drops)),
    }

    details_df = pd.DataFrame(records)
    return summary, details_df

In [35]:
hybrid_val_asr_summary, hybrid_val_asr_details = evaluate_hybrid_evasion_on_phishing(
    val_df,
    model,
    tokenizer,
    add_steps=3,
    delete_steps=5,
    max_length=512,
    max_samples=None,
    show_progress=True,
)

pd.DataFrame([{"dataset": "validation", **hybrid_val_asr_summary}])

Hybrid phishing evasion:   0%|          | 0/2610 [00:00<?, ?it/s]

,dataset,attack,n_true_phishing,n_orig_correct_phishing,n_flipped,attack_success_rate,robust_recall_on_orig_correct_phishing,avg_total_steps_used,avg_add_steps_used,avg_delete_steps_used,avg_prob_drop
0,validation,hybrid_fast_add3_delete5,2640,2610,227,0.086973,0.913027,6.171648,2.926437,3.245211,0.07002


In [36]:
hybrid_test_asr_summaries = []
hybrid_test_asr_details = {}

for i, test_df in enumerate(test_dfs, start=1):
    test_name = test_df["dataset"].iloc[0] if "dataset" in test_df.columns else f"test_{i}"

    summary, details = evaluate_hybrid_evasion_on_phishing(
        test_df,
        model,
        tokenizer,
        add_steps=3,
        delete_steps=5,
        max_length=512,
        max_samples=None,
        show_progress=True,
    )

    hybrid_test_asr_summaries.append({
        "dataset": test_name,
        **summary
    })
    hybrid_test_asr_details[test_name] = details

hybrid_test_asr_summary_df = pd.DataFrame(hybrid_test_asr_summaries)
hybrid_test_asr_summary_df

Hybrid phishing evasion:   0%|          | 0/12737 [00:00<?, ?it/s]

Hybrid phishing evasion:   0%|          | 0/1533 [00:00<?, ?it/s]

Hybrid phishing evasion:   0%|          | 0/3081 [00:00<?, ?it/s]

Hybrid phishing evasion:   0%|          | 0/1089 [00:00<?, ?it/s]

,dataset,attack,n_true_phishing,n_orig_correct_phishing,n_flipped,attack_success_rate,robust_recall_on_orig_correct_phishing,avg_total_steps_used,avg_add_steps_used,avg_delete_steps_used,avg_prob_drop
0,CEAS_08_cleaned,hybrid_fast_add3_delete5,21842,12737,10124,0.794850,0.205150,2.544791,1.832692,0.712099,0.661333
1,Nazario_cleaned,hybrid_fast_add3_delete5,1565,1533,138,0.090020,0.909980,5.598826,2.926941,2.671885,0.072030
2,Nigerian_Fraud_cleaned,hybrid_fast_add3_delete5,3332,3081,1107,0.359299,0.640701,4.758845,2.554365,2.204479,0.291398
3,SpamAssasin_cleaned,hybrid_fast_add3_delete5,1718,1089,481,0.441690,0.558310,4.394858,2.447199,1.947658,0.346789


In [ ]:
sample_text = val_df.iloc[0]["text"]

token_scores = get_token_saliency(sample_text, model, tokenizer)
word_scores = merge_wordpiece_tokens(token_scores)

pd.DataFrame(
    sorted(word_scores, key=lambda x: x[1], reverse=True)[:20],
    columns=["word", "saliency"]
)

,word,saliency
0,mail,9.971901
1,-,7.985861
2,e,4.798193
3,documentos,2.749177
4,documentos,2.608819
5,para,2.527359
6,seu,2.161697
7,uma,2.052487
8,digital,1.507084
9,tenham,1.268662


In [ ]:
import random
import numpy as np
import pandas as pd
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, roc_auc_score

def benign_prefix_attack(text):
    prefix = (
        "This is a routine internal business communication from the IT department. "
        "This message is safe, legitimate, and part of standard company operations.\n\n"
    )
    return prefix + text

def benign_suffix_attack(text):
    suffix = (
        "\n\nThis is a routine internal message. "
        "Please contact the internal IT help desk if you have any questions."
    )
    return text + suffix

def contradiction_attack(text):
    prefix = (
        "No action is required at this time. "
        "This message is only informational and does not require account verification.\n\n"
    )
    return prefix + text

def training_context_attack(text):
    prefix = (
        "This is a simulated phishing-awareness training email used for internal education. "
        "It is safe and authorized.\n\n"
    )
    return prefix + text

def noise_injection_attack(text, repeat=3):
    noise = (
        "This is a general update regarding internal documentation, procedures, "
        "and routine operational processes. "
    )
    return (noise * repeat) + text

In [ ]:
def iterative_addition_attack(
    text,
    model,
    tokenizer,
    max_steps=5,
    strategy="random",
    seed=42,
):
    """
    Iteratively adds benign / contradictory / contextual text until the model flips.

    Returns a dictionary with:
      - attacked_text
      - flipped
      - steps
      - original_pred
      - new_pred
      - original_prob
      - new_prob
      - history
    """

    rng = random.Random(seed)

    attack_functions = {
        "benign_prefix": benign_prefix_attack,
        "benign_suffix": benign_suffix_attack,
        "contradiction": contradiction_attack,
        "training_context": training_context_attack,
        "noise_injection": noise_injection_attack,
    }

    ordered_fns = [
        ("benign_prefix", benign_prefix_attack),
        ("benign_suffix", benign_suffix_attack),
        ("contradiction", contradiction_attack),
        ("training_context", training_context_attack),
        ("noise_injection", noise_injection_attack),
    ]

    original_pred, original_prob = predict_one(text, model, tokenizer)
    current_text = text
    history = []

    for step in range(1, max_steps + 1):
        if strategy == "random":
            attack_name, attack_fn = rng.choice(ordered_fns)

        elif strategy == "fixed":
            attack_name, attack_fn = ordered_fns[(step - 1) % len(ordered_fns)]

        else:
            raise ValueError("strategy must be 'random' or 'fixed'")

        current_text = attack_fn(current_text)
        new_pred, new_prob = predict_one(current_text, model, tokenizer)

        history.append({
            "step": step,
            "attack_name": attack_name,
            "pred": new_pred,
            "phishing_prob": new_prob,
            "text_preview": current_text[:300]
        })

        if new_pred != original_pred:
            return {
                "attacked_text": current_text,
                "flipped": True,
                "steps": step,
                "original_pred": original_pred,
                "new_pred": new_pred,
                "original_prob": original_prob,
                "new_prob": new_prob,
                "history": history,
            }

    final_pred, final_prob = predict_one(current_text, model, tokenizer)

    return {
        "attacked_text": current_text,
        "flipped": False,
        "steps": max_steps,
        "original_pred": original_pred,
        "new_pred": final_pred,
        "original_prob": original_prob,
        "new_prob": final_prob,
        "history": history,
    }

In [ ]:
sample_text = test_df.iloc[0]["text"]

result = iterative_addition_attack(
    sample_text,
    model,
    tokenizer,
    max_steps=5,
    strategy="fixed",
    seed=42,
)

print("Original prediction:", result["original_pred"])
print("New prediction:", result["new_pred"])
print("Flipped:", result["flipped"])
print("Steps:", result["steps"])
print("Original phishing prob:", result["original_prob"])
print("New phishing prob:", result["new_prob"])

pd.DataFrame(result["history"])

Original prediction: 0
New prediction: 1
Flipped: True
Steps: 3
Original phishing prob: 0.0008727670647203922
New phishing prob: 0.929248034954071


,step,attack_name,pred,phishing_prob,text_preview
0,1,benign_prefix,0,0.002964,This is a routine internal business communicat...
1,2,benign_suffix,0,0.010509,This is a routine internal business communicat...
2,3,contradiction,1,0.929248,No action is required at this time. This messa...


In [ ]:
def evaluate_iterative_addition_attack(
    df_eval,
    model,
    tokenizer,
    max_samples=None,
    max_steps=5,
    strategy="random",
    seed=42,
):
    """
    Evaluates iterative addition attack over a dataframe with columns:
      - text
      - label

    Returns:
      - summary dict
      - per-example dataframe
    """

    if max_samples is not None:
        df_local = df_eval.iloc[:max_samples].copy()
    else:
        df_local = df_eval.copy()

    y_true = []
    y_pred_after = []
    y_prob_after = []

    flips = 0
    steps_used = []
    records = []

    for i, row in df_local.iterrows():
        text = row["text"]
        label = int(row["label"])

        result = iterative_addition_attack(
            text=text,
            model=model,
            tokenizer=tokenizer,
            max_steps=max_steps,
            strategy=strategy,
            seed=seed + i,
        )

        y_true.append(label)
        y_pred_after.append(result["new_pred"])
        y_prob_after.append(result["new_prob"])
        steps_used.append(result["steps"])

        if result["flipped"]:
            flips += 1

        records.append({
            "label": label,
            "original_pred": result["original_pred"],
            "new_pred": result["new_pred"],
            "flipped": result["flipped"],
            "steps": result["steps"],
            "original_prob": result["original_prob"],
            "new_prob": result["new_prob"],
            "original_text": text[:300],
            "attacked_text": result["attacked_text"][:300],
        })

    precision, recall, f1, _ = precision_recall_fscore_support(
        y_true,
        y_pred_after,
        average="binary",
        zero_division=0
    )
    acc = accuracy_score(y_true, y_pred_after)

    try:
        roc_auc = roc_auc_score(y_true, y_prob_after)
    except:
        roc_auc = float("nan")

    summary = {
        "attack": f"iterative_addition_{max_steps}steps",
        "accuracy": acc,
        "precision": precision,
        "recall": recall,
        "f1": f1,
        "roc_auc": roc_auc,
        "attack_success_rate": flips / len(df_local),
        "avg_steps_used": float(np.mean(steps_used)),
    }

    details_df = pd.DataFrame(records)
    return summary, details_df

In [ ]:
iter_add_val_summary, iter_add_val_details = evaluate_iterative_addition_attack(
    val_df,
    model,
    tokenizer,
    max_samples=300,
    max_steps=5,
    strategy="random",
    seed=42,
)

pd.DataFrame([iter_add_val_summary])

Iterative addition eval:   0%|          | 0/300 [00:00<?, ?it/s]

,attack,accuracy,precision,recall,f1,roc_auc,attack_success_rate,avg_steps_used,n_samples
0,iterative_addition_5steps,0.956667,0.9,0.980198,0.938389,0.998308,0.046667,4.896667,300


In [ ]:
iter_add_test_summary, iter_add_test_details = evaluate_iterative_addition_attack(
    test_df,
    model,
    tokenizer,
    max_samples=300,
    max_steps=5,
    strategy="random",
    seed=42,
)

pd.DataFrame([iter_add_test_summary])

Iterative addition eval:   0%|          | 0/300 [00:00<?, ?it/s]

,attack,accuracy,precision,recall,f1,roc_auc,attack_success_rate,avg_steps_used,n_samples
0,iterative_addition_5steps,0.61,0.167883,0.884615,0.282209,0.888195,0.353333,4.06,300


In [ ]:
iter_add_val_summary["dataset"] = "validation"
iter_add_test_summary["dataset"] = "test"

iter_add_results_df = pd.DataFrame([
    iter_add_val_summary,
    iter_add_test_summary,
])

iter_add_results_df

,attack,accuracy,precision,recall,f1,roc_auc,attack_success_rate,avg_steps_used,n_samples,dataset
0,iterative_addition_5steps,0.956667,0.900000,0.980198,0.938389,0.998308,0.046667,4.896667,300,validation
1,iterative_addition_5steps,0.610000,0.167883,0.884615,0.282209,0.888195,0.353333,4.060000,300,test


In [ ]:
iter_add_test_details[iter_add_test_details["flipped"] == True].head(10)

,label,original_pred,new_pred,flipped,steps,original_prob,new_prob,original_text,attacked_text
0,0,0,1,True,3,0.000873,0.712182,Subject: Feedback sulla collaborazione con sol...,No action is required at this time. This messa...
2,0,1,0,True,1,0.507598,0.048137,Subject: Exploring Collaboration Opportunities...,This is a simulated phishing-awareness trainin...
3,0,0,1,True,1,0.037117,0.846387,Subject: Clarifications on e-commerce app enha...,No action is required at this time. This messa...
4,0,0,1,True,2,0.152537,0.801159,Subject: Opportunity for Collaboration on Digi...,This is a simulated phishing-awareness trainin...
6,0,0,1,True,3,0.000989,0.516409,Subject: Confirmation of Meeting with SmithTec...,No action is required at this time. This messa...
9,0,1,0,True,2,0.715623,0.067917,Subject: Internal Meeting: Online Accessories ...,This is a general update regarding internal do...
11,0,1,0,True,1,0.514705,0.001595,Subject: Request for Technical Specifications ...,This is a general update regarding internal do...
12,0,1,0,True,3,0.973266,0.276935,Subject: Company Performance Update and Future...,This is a general update regarding internal do...
14,0,0,1,True,5,0.026734,0.805326,Subject: Exploring the New Inventory System Fe...,No action is required at this time. This messa...
21,0,0,1,True,1,0.432693,0.617051,Subject: Request for Information on New Eco-Fr...,This is a simulated phishing-awareness trainin...


In [ ]:
def iterative_addition_attack_greedy(
    text,
    model,
    tokenizer,
    max_steps=5,
):
    attack_functions = [
        ("benign_prefix", benign_prefix_attack),
        ("benign_suffix", benign_suffix_attack),
        ("contradiction", contradiction_attack),
        ("training_context", training_context_attack),
        ("noise_injection", noise_injection_attack),
    ]

    original_pred, original_prob = predict_one(text, model, tokenizer)
    current_text = text
    history = []

    for step in range(1, max_steps + 1):
        best_text = None
        best_name = None
        best_pred = None
        best_prob = None

        # choose the addition that most reduces phishing probability
        for attack_name, attack_fn in attack_functions:
            candidate_text = attack_fn(current_text)
            candidate_pred, candidate_prob = predict_one(candidate_text, model, tokenizer)

            if best_prob is None or candidate_prob < best_prob:
                best_text = candidate_text
                best_name = attack_name
                best_pred = candidate_pred
                best_prob = candidate_prob

        current_text = best_text

        history.append({
            "step": step,
            "attack_name": best_name,
            "pred": best_pred,
            "phishing_prob": best_prob,
            "text_preview": current_text[:300],
        })

        if best_pred != original_pred:
            return {
                "attacked_text": current_text,
                "flipped": True,
                "steps": step,
                "original_pred": original_pred,
                "new_pred": best_pred,
                "original_prob": original_prob,
                "new_prob": best_prob,
                "history": history,
            }

    final_pred, final_prob = predict_one(current_text, model, tokenizer)

    return {
        "attacked_text": current_text,
        "flipped": False,
        "steps": max_steps,
        "original_pred": original_pred,
        "new_pred": final_pred,
        "original_prob": original_prob,
        "new_prob": final_prob,
        "history": history,
    }

In [ ]:
def iterative_addition_attack_greedy_fast(
    text,
    model,
    tokenizer,
    max_steps=5,
):
    attack_functions = [
        ("benign_prefix", benign_prefix_attack),
        ("benign_suffix", benign_suffix_attack),
        ("contradiction", contradiction_attack),
        ("training_context", training_context_attack),
        ("noise_injection", noise_injection_attack),
    ]

    original_pred, original_prob = predict_one_fast(text, model, tokenizer)
    current_text = text
    history = []

    for step in range(1, max_steps + 1):
        candidate_names = []
        candidate_texts = []

        for attack_name, attack_fn in attack_functions:
            candidate_names.append(attack_name)
            candidate_texts.append(attack_fn(current_text))

        candidate_preds, candidate_probs = batch_predict_fast(
            candidate_texts, model, tokenizer, batch_size=len(candidate_texts)
        )

        best_idx = int(np.argmin(candidate_probs))
        best_text = candidate_texts[best_idx]
        best_name = candidate_names[best_idx]
        best_pred = candidate_preds[best_idx]
        best_prob = candidate_probs[best_idx]

        current_text = best_text

        history.append({
            "step": step,
            "attack_name": best_name,
            "pred": best_pred,
            "phishing_prob": best_prob,
            "text_preview": current_text[:300],
        })

        if best_pred != original_pred:
            return {
                "attacked_text": current_text,
                "flipped": True,
                "steps": step,
                "original_pred": original_pred,
                "new_pred": best_pred,
                "original_prob": original_prob,
                "new_prob": best_prob,
                "history": history,
            }

    final_pred, final_prob = predict_one_fast(current_text, model, tokenizer)

    return {
        "attacked_text": current_text,
        "flipped": False,
        "steps": max_steps,
        "original_pred": original_pred,
        "new_pred": final_pred,
        "original_prob": original_prob,
        "new_prob": final_prob,
        "history": history,
    }

In [ ]:
greedy_result = iterative_addition_attack_greedy(
    test_df.iloc[0]["text"],
    model,
    tokenizer,
    max_steps=5,
)

pd.DataFrame(greedy_result["history"])

,step,attack_name,pred,phishing_prob,text_preview
0,1,noise_injection,0,0.000440,This is a general update regarding internal do...
1,2,noise_injection,0,0.000328,This is a general update regarding internal do...
2,3,noise_injection,0,0.000277,This is a general update regarding internal do...
3,4,noise_injection,0,0.000295,This is a general update regarding internal do...
4,5,noise_injection,0,0.000298,This is a general update regarding internal do...


Error: Runtime no longer has a reference to this dataframe, please re-run this cell and try again.


In [ ]:
from tqdm.auto import tqdm

def evaluate_iterative_addition_attack(
    df_eval,
    model,
    tokenizer,
    max_samples=None,
    max_steps=5,
    strategy="random",
    seed=42,
    show_progress=True,
):
    if max_samples is not None:
        df_local = df_eval.iloc[:max_samples].copy()
    else:
        df_local = df_eval.copy()

    y_true = []
    y_pred_after = []
    y_prob_after = []

    flips = 0
    steps_used = []
    records = []

    iterator = df_local.iterrows()
    if show_progress:
        iterator = tqdm(iterator, total=len(df_local), desc="Iterative addition eval")

    for i, row in iterator:
        text = row["text"]
        label = int(row["label"])

        result = iterative_addition_attack(
            text=text,
            model=model,
            tokenizer=tokenizer,
            max_steps=max_steps,
            strategy=strategy,
            seed=seed + i,
        )

        y_true.append(label)
        y_pred_after.append(result["new_pred"])
        y_prob_after.append(result["new_prob"])
        steps_used.append(result["steps"])

        if result["flipped"]:
            flips += 1

        records.append({
            "label": label,
            "original_pred": result["original_pred"],
            "new_pred": result["new_pred"],
            "flipped": result["flipped"],
            "steps": result["steps"],
            "original_prob": result["original_prob"],
            "new_prob": result["new_prob"],
            "original_text": text[:300],
            "attacked_text": result["attacked_text"][:300],
        })

    precision, recall, f1, _ = precision_recall_fscore_support(
        y_true,
        y_pred_after,
        average="binary",
        zero_division=0
    )
    acc = accuracy_score(y_true, y_pred_after)

    try:
        roc_auc = roc_auc_score(y_true, y_prob_after)
    except:
        roc_auc = float("nan")

    summary = {
        "attack": f"iterative_addition_{max_steps}steps",
        "accuracy": acc,
        "precision": precision,
        "recall": recall,
        "f1": f1,
        "roc_auc": roc_auc,
        "attack_success_rate": flips / len(df_local),
        "avg_steps_used": float(np.mean(steps_used)),
        "n_samples": len(df_local),
    }

    details_df = pd.DataFrame(records)
    return summary, details_df

In [ ]:
iter_add_val_summary_full, iter_add_val_details_full = evaluate_iterative_addition_attack(
    val_df, model, tokenizer, max_samples=None, max_steps=5, strategy="random", seed=42
)

iter_add_test_summary_full, iter_add_test_details_full = evaluate_iterative_addition_attack(
    test_df, model, tokenizer, max_samples=None, max_steps=5, strategy="random", seed=42
)

pd.DataFrame([
    {"dataset": "validation", **iter_add_val_summary_full},
    {"dataset": "test", **iter_add_test_summary_full},
])

Iterative addition eval:   0%|          | 0/3960 [00:00<?, ?it/s]

Iterative addition eval:   0%|          | 0/11502 [00:00<?, ?it/s]

,dataset,attack,accuracy,precision,recall,f1,roc_auc,attack_success_rate,avg_steps_used,n_samples
0,validation,iterative_addition_5steps,0.967677,0.924948,0.985196,0.954122,0.997934,0.030556,4.932828,3960
1,test,iterative_addition_5steps,0.717875,0.691654,0.827885,0.753663,0.850767,0.289254,4.183012,11502


In [ ]:
def evaluate_iterative_addition_attack_greedy(
    df_eval,
    model,
    tokenizer,
    max_samples=None,
    max_steps=5,
    show_progress=True,
):
    if max_samples is not None:
        df_local = df_eval.iloc[:max_samples].copy()
    else:
        df_local = df_eval.copy()

    y_true = []
    y_pred_after = []
    y_prob_after = []

    flips = 0
    steps_used = []
    records = []

    iterator = df_local.iterrows()
    if show_progress:
        iterator = tqdm(iterator, total=len(df_local), desc="Greedy iterative addition eval")

    for _, row in iterator:
        text = row["text"]
        label = int(row["label"])

        result = iterative_addition_attack_greedy(
            text=text,
            model=model,
            tokenizer=tokenizer,
            max_steps=max_steps,
        )

        y_true.append(label)
        y_pred_after.append(result["new_pred"])
        y_prob_after.append(result["new_prob"])
        steps_used.append(result["steps"])

        if result["flipped"]:
            flips += 1

        records.append({
            "label": label,
            "original_pred": result["original_pred"],
            "new_pred": result["new_pred"],
            "flipped": result["flipped"],
            "steps": result["steps"],
            "original_prob": result["original_prob"],
            "new_prob": result["new_prob"],
            "original_text": text[:300],
            "attacked_text": result["attacked_text"][:300],
        })

    precision, recall, f1, _ = precision_recall_fscore_support(
        y_true,
        y_pred_after,
        average="binary",
        zero_division=0
    )
    acc = accuracy_score(y_true, y_pred_after)

    try:
        roc_auc = roc_auc_score(y_true, y_prob_after)
    except:
        roc_auc = float("nan")

    summary = {
        "attack": f"iterative_addition_greedy_{max_steps}steps",
        "accuracy": acc,
        "precision": precision,
        "recall": recall,
        "f1": f1,
        "roc_auc": roc_auc,
        "attack_success_rate": flips / len(df_local),
        "avg_steps_used": float(np.mean(steps_used)),
        "n_samples": len(df_local),
    }

    details_df = pd.DataFrame(records)
    return summary, details_df

In [ ]:
from tqdm.auto import tqdm
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, roc_auc_score
import pandas as pd
import numpy as np

def evaluate_iterative_addition_attack_greedy_fast(
    df_eval,
    model,
    tokenizer,
    max_samples=None,
    max_steps=5,
    show_progress=True,
):
    if max_samples is not None:
        df_local = df_eval.iloc[:max_samples].copy()
    else:
        df_local = df_eval.copy()

    y_true = []
    y_pred_after = []
    y_prob_after = []

    flips = 0
    steps_used = []
    records = []

    iterator = df_local.iterrows()
    if show_progress:
        iterator = tqdm(iterator, total=len(df_local), desc="Greedy iterative addition eval")

    for _, row in iterator:
        text = row["text"]
        label = int(row["label"])

        result = iterative_addition_attack_greedy_fast(
            text=text,
            model=model,
            tokenizer=tokenizer,
            max_steps=max_steps,
        )

        y_true.append(label)
        y_pred_after.append(result["new_pred"])
        y_prob_after.append(result["new_prob"])
        steps_used.append(result["steps"])

        if result["flipped"]:
            flips += 1

        records.append({
            "label": label,
            "original_pred": result["original_pred"],
            "new_pred": result["new_pred"],
            "flipped": result["flipped"],
            "steps": result["steps"],
            "original_prob": result["original_prob"],
            "new_prob": result["new_prob"],
            "original_text": text[:300],
            "attacked_text": result["attacked_text"][:300],
        })

    precision, recall, f1, _ = precision_recall_fscore_support(
        y_true,
        y_pred_after,
        average="binary",
        zero_division=0
    )
    acc = accuracy_score(y_true, y_pred_after)

    try:
        roc_auc = roc_auc_score(y_true, y_prob_after)
    except:
        roc_auc = float("nan")

    summary = {
        "attack": f"iterative_addition_greedy_fast_{max_steps}steps",
        "accuracy": acc,
        "precision": precision,
        "recall": recall,
        "f1": f1,
        "roc_auc": roc_auc,
        "attack_success_rate": flips / len(df_local),
        "avg_steps_used": float(np.mean(steps_used)),
        "n_samples": len(df_local),
    }

    details_df = pd.DataFrame(records)
    return summary, details_df

In [ ]:
greedy_val_summary, greedy_val_details = evaluate_iterative_addition_attack_greedy(
    val_df, model, tokenizer, max_samples=None, max_steps=5
)

greedy_test_summary, greedy_test_details = evaluate_iterative_addition_attack_greedy(
    test_df, model, tokenizer, max_samples=None, max_steps=5
)

pd.DataFrame([
    {"dataset": "validation", **greedy_val_summary},
    {"dataset": "test", **greedy_test_summary},
])

Greedy iterative addition eval:   0%|          | 0/3960 [00:00<?, ?it/s]

Greedy iterative addition eval:   0%|          | 0/11502 [00:00<?, ?it/s]

,dataset,attack,accuracy,precision,recall,f1,roc_auc,attack_success_rate,avg_steps_used,n_samples
0,validation,iterative_addition_greedy_5steps,0.973737,0.993666,0.928942,0.960214,0.998666,0.024495,4.940404,3960
1,test,iterative_addition_greedy_5steps,0.688141,0.976652,0.411608,0.579139,0.831201,0.283690,4.218397,11502


In [ ]:
import os
import re
import torch
import numpy as np
import pandas as pd
from tqdm.auto import tqdm
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, roc_auc_score

os.environ["TOKENIZERS_PARALLELISM"] = "true"

device = "cuda" if torch.cuda.is_available() else "cpu"
model = model.to(device)
model.eval()

torch.set_float32_matmul_precision("high")
torch.backends.cuda.matmul.allow_tf32 = True
torch.backends.cudnn.allow_tf32 = True

def predict_one_fast(text, model, tokenizer, max_length=512):
    enc = tokenizer(
        text,
        return_tensors="pt",
        truncation=True,
        max_length=max_length,
        padding=False,
    )
    enc = {k: v.to(model.device, non_blocking=True) for k, v in enc.items()}

    with torch.inference_mode():
        with torch.autocast(device_type="cuda", dtype=torch.float16, enabled=torch.cuda.is_available()):
            outputs = model(**enc)
            probs = torch.softmax(outputs.logits, dim=-1)[0]

    probs = probs.detach().float().cpu().numpy()
    pred = int(np.argmax(probs))
    return pred, float(probs[1])

def batch_predict_fast(texts, model, tokenizer, batch_size=256, max_length=512):
    preds = []
    probs_out = []

    for i in range(0, len(texts), batch_size):
        batch = texts[i:i+batch_size]

        enc = tokenizer(
            batch,
            return_tensors="pt",
            truncation=True,
            max_length=max_length,
            padding=True,
        )
        enc = {k: v.to(model.device, non_blocking=True) for k, v in enc.items()}

        with torch.inference_mode():
            with torch.autocast(device_type="cuda", dtype=torch.float16, enabled=torch.cuda.is_available()):
                outputs = model(**enc)
                probs = torch.softmax(outputs.logits, dim=-1)

        probs = probs.detach().float().cpu().numpy()
        preds.extend(np.argmax(probs, axis=1).tolist())
        probs_out.extend(probs[:, 1].tolist())

    return preds, probs_out

In [ ]:
def get_token_saliency_fast(text, model, tokenizer, max_length=512):
    model.eval()

    enc = tokenizer(
        text,
        return_tensors="pt",
        truncation=True,
        max_length=max_length,
    )

    input_ids = enc["input_ids"].to(model.device, non_blocking=True)
    attention_mask = enc["attention_mask"].to(model.device, non_blocking=True)

    embedding_layer = model.get_input_embeddings()
    inputs_embeds = embedding_layer(input_ids).detach()
    inputs_embeds.requires_grad_(True)

    model.zero_grad(set_to_none=True)

    outputs = model(inputs_embeds=inputs_embeds, attention_mask=attention_mask)
    pred_class = torch.argmax(outputs.logits, dim=1)
    score = outputs.logits[0, pred_class]
    score.backward()

    grads = inputs_embeds.grad[0]
    saliency = grads.norm(dim=1).detach().float().cpu().numpy()
    tokens = tokenizer.convert_ids_to_tokens(input_ids[0])

    return list(zip(tokens, saliency.tolist()))

In [ ]:
def merge_wordpiece_tokens(token_scores):
    words = []
    current_word = ""
    current_score = 0.0

    special_tokens = {"[CLS]", "[SEP]", "[PAD]", "<s>", "</s>", "<pad>"}

    for tok, score in token_scores:
        if tok in special_tokens:
            continue

        if tok.startswith("##"):  # BERT
            current_word += tok[2:]
            current_score += score
        elif tok.startswith("Ġ"):  # RoBERTa
            if current_word:
                words.append((current_word, current_score))
            current_word = tok[1:]
            current_score = score
        else:
            if current_word:
                words.append((current_word, current_score))
            current_word = tok
            current_score = score

    if current_word:
        words.append((current_word, current_score))

    cleaned = []
    for w, s in words:
        w = w.strip()
        w = re.sub(r"[^\w@.\-:/]", "", w)
        if w:
            cleaned.append((w, s))

    return cleaned

def important_word_deletion_attack_fast(text, model, tokenizer, k=1):
    token_scores = get_token_saliency_fast(text, model, tokenizer)
    word_scores = merge_wordpiece_tokens(token_scores)
    ranked_words = sorted(word_scores, key=lambda x: x[1], reverse=True)

    stop = {
        "subject", "body", "the", "a", "an", "and", "or", "to", "of", "in",
        "for", "on", "at", "is", "are", "this", "that", "with", "from"
    }

    targets = []
    for w, _ in ranked_words:
        wl = w.lower()
        if len(wl) >= 3 and wl not in stop and wl.isprintable():
            targets.append(wl)
        if len(targets) >= k:
            break

    attacked_words = []
    for w in text.split():
        clean = re.sub(r"^[^\w]+|[^\w]+$", "", w).lower()
        if clean in targets:
            continue
        attacked_words.append(w)

    return " ".join(attacked_words)

In [ ]:
def benign_prefix_attack(text):
    prefix = (
        "This is a routine internal business communication from the IT department. "
        "This message is safe, legitimate, and part of standard company operations.\n\n"
    )
    return prefix + text

def benign_suffix_attack(text):
    suffix = (
        "\n\nThis is a routine internal message. "
        "Please contact the internal IT help desk if you have any questions."
    )
    return text + suffix

def contradiction_attack(text):
    prefix = (
        "No action is required at this time. "
        "This message is only informational and does not require account verification.\n\n"
    )
    return prefix + text

def training_context_attack(text):
    prefix = (
        "This is a simulated phishing-awareness training email used for internal education. "
        "It is safe and authorized.\n\n"
    )
    return prefix + text

def noise_injection_attack(text, repeat=3):
    noise = (
        "This is a general update regarding internal documentation, procedures, "
        "and routine operational processes. "
    )
    return (noise * repeat) + text

In [19]:
def hybrid_add_then_delete_attack_fast(
    text,
    model,
    tokenizer,
    add_steps=3,
    delete_steps=5,
    max_length=512,
):
    """
    Phase 1: greedy batched additions
    Phase 2: saliency-based deletions

    Returns:
      attacked_text, flipped, total_steps, original/new pred and prob, history
    """

    original_pred, original_prob = predict_one_fast(text, model, tokenizer, max_length=max_length)
    current_text = text
    history = []

    addition_fns = [
        ("benign_prefix", benign_prefix_attack),
        ("benign_suffix", benign_suffix_attack),
        ("contradiction", contradiction_attack),
        ("training_context", training_context_attack),
        ("noise_injection", noise_injection_attack),
    ]

    # Phase 1: greedy additions, scored in one batch per step
    for step in range(1, add_steps + 1):
        candidate_names = []
        candidate_texts = []

        for attack_name, attack_fn in addition_fns:
            candidate_names.append(attack_name)
            candidate_texts.append(attack_fn(current_text))

        candidate_preds, candidate_probs = batch_predict_fast(
            candidate_texts,
            model,
            tokenizer,
            batch_size=len(candidate_texts),
            max_length=max_length,
        )

        best_idx = int(np.argmin(candidate_probs))
        current_text = candidate_texts[best_idx]
        best_name = candidate_names[best_idx]
        best_pred = candidate_preds[best_idx]
        best_prob = candidate_probs[best_idx]

        history.append({
            "phase": "add",
            "step": step,
            "attack_name": best_name,
            "pred": best_pred,
            "phishing_prob": best_prob,
            "text_preview": current_text[:300],
        })

        if best_pred != original_pred:
            return {
                "attacked_text": current_text,
                "flipped": True,
                "steps_add": step,
                "steps_delete": 0,
                "total_steps": step,
                "original_pred": original_pred,
                "new_pred": best_pred,
                "original_prob": original_prob,
                "new_prob": best_prob,
                "history": history,
            }

    # Phase 2: saliency deletion
    for dstep in range(1, delete_steps + 1):
        new_text = important_word_deletion_attack_fast(
            current_text,
            model,
            tokenizer,
            k=1
        )

        if new_text == current_text:
            break

        current_text = new_text
        new_pred, new_prob = predict_one_fast(current_text, model, tokenizer, max_length=max_length)

        history.append({
            "phase": "delete",
            "step": dstep,
            "attack_name": "important_word_delete_k1",
            "pred": new_pred,
            "phishing_prob": new_prob,
            "text_preview": current_text[:300],
        })

        if new_pred != original_pred:
            return {
                "attacked_text": current_text,
                "flipped": True,
                "steps_add": add_steps,
                "steps_delete": dstep,
                "total_steps": add_steps + dstep,
                "original_pred": original_pred,
                "new_pred": new_pred,
                "original_prob": original_prob,
                "new_prob": new_prob,
                "history": history,
            }

    final_pred, final_prob = predict_one_fast(current_text, model, tokenizer, max_length=max_length)

    return {
        "attacked_text": current_text,
        "flipped": False,
        "steps_add": add_steps,
        "steps_delete": delete_steps,
        "total_steps": add_steps + delete_steps,
        "original_pred": original_pred,
        "new_pred": final_pred,
        "original_prob": original_prob,
        "new_prob": final_prob,
        "history": history,
    }

In [ ]:
def evaluate_hybrid_add_then_delete_attack_fast(
    df_eval,
    model,
    tokenizer,
    max_samples=None,
    add_steps=3,
    delete_steps=5,
    max_length=512,
    show_progress=True,
):
    if max_samples is not None:
        df_local = df_eval.iloc[:max_samples].copy()
    else:
        df_local = df_eval.copy()

    y_true = []
    y_pred_after = []
    y_prob_after = []

    flips = 0
    total_steps_used = []
    add_steps_used = []
    delete_steps_used = []
    records = []

    iterator = df_local.iterrows()
    if show_progress:
        iterator = tqdm(iterator, total=len(df_local), desc="Hybrid fast eval")

    for _, row in iterator:
        text = row["text"]
        label = int(row["label"])

        result = hybrid_add_then_delete_attack_fast(
            text=text,
            model=model,
            tokenizer=tokenizer,
            add_steps=add_steps,
            delete_steps=delete_steps,
            max_length=max_length,
        )

        y_true.append(label)
        y_pred_after.append(result["new_pred"])
        y_prob_after.append(result["new_prob"])

        total_steps_used.append(result["total_steps"])
        add_steps_used.append(result["steps_add"])
        delete_steps_used.append(result["steps_delete"])

        if result["flipped"]:
            flips += 1

        records.append({
            "label": label,
            "original_pred": result["original_pred"],
            "new_pred": result["new_pred"],
            "flipped": result["flipped"],
            "steps_add": result["steps_add"],
            "steps_delete": result["steps_delete"],
            "total_steps": result["total_steps"],
            "original_prob": result["original_prob"],
            "new_prob": result["new_prob"],
            "original_text": text[:300],
            "attacked_text": result["attacked_text"][:300],
        })

    precision, recall, f1, _ = precision_recall_fscore_support(
        y_true,
        y_pred_after,
        average="binary",
        zero_division=0
    )
    acc = accuracy_score(y_true, y_pred_after)

    try:
        roc_auc = roc_auc_score(y_true, y_prob_after)
    except:
        roc_auc = float("nan")

    summary = {
        "attack": f"hybrid_fast_add{add_steps}_delete{delete_steps}",
        "accuracy": acc,
        "precision": precision,
        "recall": recall,
        "f1": f1,
        "roc_auc": roc_auc,
        "attack_success_rate": flips / len(df_local),
        "avg_total_steps_used": float(np.mean(total_steps_used)),
        "avg_add_steps_used": float(np.mean(add_steps_used)),
        "avg_delete_steps_used": float(np.mean(delete_steps_used)),
        "n_samples": len(df_local),
    }

    details_df = pd.DataFrame(records)
    return summary, details_df

In [ ]:
hybrid_val_summary_fast, hybrid_val_details_fast = evaluate_hybrid_add_then_delete_attack_fast(
    val_df,
    model,
    tokenizer,
    max_samples=None,
    add_steps=3,
    delete_steps=5,
    max_length=512,
    show_progress=True,
)

hybrid_test_summary_fast, hybrid_test_details_fast = evaluate_hybrid_add_then_delete_attack_fast(
    test_df,
    model,
    tokenizer,
    max_samples=None,
    add_steps=3,
    delete_steps=5,
    max_length=512,
    show_progress=True,
)

pd.DataFrame([
    {"dataset": "validation", **hybrid_val_summary_fast},
    {"dataset": "test", **hybrid_test_summary_fast},
])

Hybrid fast eval:   0%|          | 0/3960 [00:00<?, ?it/s]

Hybrid fast eval:   0%|          | 0/11502 [00:00<?, ?it/s]

,dataset,attack,accuracy,precision,recall,f1,roc_auc,attack_success_rate,avg_total_steps_used,avg_add_steps_used,avg_delete_steps_used,n_samples
0,validation,hybrid_fast_add3_delete5,0.972222,0.98974,0.928201,0.957983,0.998193,0.027020,7.864141,2.980051,4.884091,3960
1,test,hybrid_fast_add3_delete5,0.706660,0.97023,0.451134,0.615893,0.852527,0.295775,6.352113,2.672666,3.679447,11502


In [ ]:
hybrid_test_details_fast[hybrid_test_details_fast["flipped"] == True].head(10)